# AA-510 Agentic AI Systems (Group 4)

Team Member: Jeffery Smith、Jinyuan He、Mostafa Zamaniturk

**Part A: Agent Deliverable - Technical Artifacts**
- A notebook that contains your data pipeline (DE is the owner)
- A notebook that shows your agent definition (AIE is the owner)
- Review the rubric to determine agent requirements.
- An example of five traces, including the evaluation of the agent (AIE is the owner)
- Traces should be presented via an established provider. Databricks will leverage MLflow by default; alternatives are Arize Phoenix, LangSmith, and Braintrust, among others.
- There should be at least 2 different LLMs used within the agent on the same trace to compare performance (this counts as one trace per the five above).
- Evaluation can be an LLM judge (strongly encouraged) or manual.
- Provide commentary in a notebook cell that talks about the agent performance, and what the evaluation showed, including a comparison between the agent using different LLMs.
- Show 2 examples of the agent gracefully rejecting an irrelevant user input.

# 1. Project Overview

### 1.1 Business problem
### 1.2 Agent goal
### 1.3 Team roles: PM / DE / AIE

# 2. Data Pipeline

### 2.1 Load dataset

In [0]:
%pip install -U -qqqq kaggle backoff databricks-openai uv databricks-agents "mlflow>=3.9" databricks-mcp nest_asyncio "gepa>=0.1.0" "databricks-vectorsearch<0.74"
%pip install -U  sentence-transformers
%pip install duckduckgo-search 
dbutils.library.restartPython()

In [0]:
# dbutils.library.restartPython()

In [0]:
# Download dataset from Kaggle
import os
import pandas as pd

# Create directory if it doesn't exist
os.makedirs('/tmp/ai_reviews', exist_ok=True)

# Download to local tmp directory
! kaggle datasets download -d jahnavikachhia23/the-generative-ai-ecosystem-50k-user-reviews-2026 --unzip -p /tmp/ai_reviews

# Load the CSV into pandas first
pandas_df = pd.read_csv('/tmp/ai_reviews/The_ Generative_AI_Ecosystem_50k_User_Reviews_2026.csv')

# Convert to Spark DataFrame
df = spark.createDataFrame(pandas_df)

display(df)

In [0]:
# Create new catalog "group4"
spark.sql("CREATE CATALOG IF NOT EXISTS group4")

# Save the dataset as a table in the new catalog (using full 3-part name)
df.write.format("delta").mode("overwrite").saveAsTable("group4.default.ai_reviews")

### 2.2 Exploratory Data Analysis

In [0]:
# Exploratory Data Analysis (EDA) on ai_reviews dataset

# Show schema
df.printSchema()

# Show summary statistics for numeric columns
display(df.describe())

# Show first 10 rows
display(df.limit(10))

# Count total rows
row_count = df.count()

# Show distribution of review ratings (if column exists)
if 'rating' in df.columns:
    display(df.groupBy('rating').count().orderBy('rating'))

# Show top 10 most frequent products (if column exists)
if 'product_name' in df.columns:
    display(df.groupBy('product_name').count().orderBy('count', ascending=False).limit(10))

# Show null counts per column
from pyspark.sql.functions import col, sum as spark_sum
null_counts = df.select([spark_sum(col(c).isNull().cast("int")).alias(c) for c in df.columns])
display(null_counts)

In [0]:
# Aggregate Table - Reviews by Product Sentiment Categories
from pyspark.sql.functions import when, col

# Load the base table
df = spark.table("group4.default.ai_reviews")

# Create sentiment categories based on Sentiment_Polarity
# Positive: > 0.3, Neutral: -0.3 to 0.3, Negative: < -0.3
df_with_sentiment = df.withColumn(
    "sentiment_category",
    when(col("Sentiment_Polarity") > 0.3, "Positive")
    .when(col("Sentiment_Polarity") < -0.3, "Negative")
    .otherwise("Neutral")
)

# Show distribution of sentiment categories
print("Sentiment Category Distribution:")
display(df_with_sentiment.groupBy("sentiment_category").count().orderBy("count", ascending=False))

In [0]:
# Feature Engineering: Temporal Features
from pyspark.sql.functions import to_timestamp, year, month, dayofmonth, dayofweek, hour, quarter, weekofyear

# Parse Review_Date and extract temporal features
df_temporal = df_with_sentiment.withColumn(
    "review_timestamp", to_timestamp(col("Review_Date"), "yyyy-MM-dd HH:mm:ss")
).withColumn(
    "review_year", year(col("review_timestamp"))
).withColumn(
    "review_month", month(col("review_timestamp"))
).withColumn(
    "review_day", dayofmonth(col("review_timestamp"))
).withColumn(
    "review_dayofweek", dayofweek(col("review_timestamp"))  # 1=Sunday, 7=Saturday
).withColumn(
    "review_hour", hour(col("review_timestamp"))
).withColumn(
    "review_quarter", quarter(col("review_timestamp"))
).withColumn(
    "review_week", weekofyear(col("review_timestamp"))
)

# Show sample with temporal features
print("Sample data with temporal features:")
display(df_temporal.select(
    "App", "Review_Date", "review_year", "review_month", "review_day", 
    "review_dayofweek", "review_hour", "Star_Rating"
).limit(10))

In [0]:
# Aggregate Table: Reviews by Product (App)
from pyspark.sql.functions import count, avg, sum as spark_sum, min as spark_min, max as spark_max

# Aggregate by product (App)
df_product_agg = df_temporal.groupBy("App").agg(
    count("*").alias("total_reviews"),
    avg("Star_Rating").alias("avg_rating"),
    avg("Sentiment_Polarity").alias("avg_sentiment"),
    avg("Word_Count").alias("avg_word_count"),
    spark_sum("Thumbs_Up_Count").alias("total_thumbs_up"),
    spark_min("review_timestamp").alias("first_review_date"),
    spark_max("review_timestamp").alias("latest_review_date"),
    count(when(col("sentiment_category") == "Positive", 1)).alias("positive_reviews"),
    count(when(col("sentiment_category") == "Negative", 1)).alias("negative_reviews"),
    count(when(col("sentiment_category") == "Neutral", 1)).alias("neutral_reviews")
).orderBy("total_reviews", ascending=False)

print("Reviews aggregated by Product (App):")
display(df_product_agg)

In [0]:
# Aggregate Table: Reviews by Date
from pyspark.sql.functions import date_format

# Create date column (without time)
df_with_date = df_temporal.withColumn(
    "review_date_only", date_format(col("review_timestamp"), "yyyy-MM-dd")
)

# Aggregate by date
df_date_agg = df_with_date.groupBy("review_date_only").agg(
    count("*").alias("total_reviews"),
    avg("Star_Rating").alias("avg_rating"),
    avg("Sentiment_Polarity").alias("avg_sentiment"),
    spark_sum("Thumbs_Up_Count").alias("total_thumbs_up"),
    count(when(col("sentiment_category") == "Positive", 1)).alias("positive_reviews"),
    count(when(col("sentiment_category") == "Negative", 1)).alias("negative_reviews")
).orderBy("review_date_only", ascending=False)

print("Reviews aggregated by Date:")
display(df_date_agg.limit(20))

# Save aggregate table
df_date_agg.write.format("delta").mode("overwrite").saveAsTable("group4.default.reviews_by_date")
print("\nAggregate table saved to group4.default.reviews_by_date")

In [0]:
# Aggregate by star rating
df_rating_agg = df_temporal.groupBy("Star_Rating").agg(
    count("*").alias("total_reviews"),
    avg("Sentiment_Polarity").alias("avg_sentiment"),
    avg("Word_Count").alias("avg_word_count"),
    avg("Review_Length_Chars").alias("avg_review_length"),
    spark_sum("Thumbs_Up_Count").alias("total_thumbs_up")
).orderBy("Star_Rating")

print("Reviews aggregated by Star Rating:")
display(df_rating_agg)

# Save aggregate table
df_rating_agg.write.format("delta").mode("overwrite").saveAsTable("group4.default.reviews_by_rating")
print("\nAggregate table saved to group4.default.reviews_by_rating")

In [0]:
# Product Category Hierarchy
from pyspark.sql.functions import lit

# Create product category hierarchy based on App names
# Categorize AI apps into types
df_categorized = df_temporal.withColumn(
    "app_category",
    when(col("App").isin(["ChatGPT", "Claude", "Gemini", "Copilot"]), "Conversational AI")
    .when(col("App").isin(["Perplexity", "You.com"]), "Search AI")
    .when(col("App").isin(["Midjourney", "DALL-E", "Stable Diffusion"]), "Image Generation")
    .otherwise("Other AI")
).withColumn(
    "app_provider",
    when(col("App") == "ChatGPT", "OpenAI")
    .when(col("App") == "Claude", "Anthropic")
    .when(col("App") == "Gemini", "Google")
    .when(col("App") == "Copilot", "Microsoft")
    .when(col("App") == "Perplexity", "Perplexity AI")
    .otherwise("Other")
)

# Show distribution by category
print("Distribution by App Category:")
display(df_categorized.groupBy("app_category", "App").count().orderBy("app_category", "count", ascending=False))

In [0]:
# Review theme Hierarchy
from pyspark.sql.functions import lit

# Show distribution by Review_Theme
print("Distribution by App Category:")
display(df_categorized.groupBy("Review_Theme").count().orderBy("Review_Theme", "count", ascending=False))

### 2.3 Create embeddings

In [0]:

from pyspark.sql.functions import pandas_udf, col
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, ArrayType, FloatType
import pandas as pd
from sentence_transformers import SentenceTransformer

# import model
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
df = spark.table("group4.default.ai_reviews")


In [0]:
# Process embeddings in batches on the driver to avoid worker download issues
# The model is already loaded in cell 19

import pandas as pd
from pyspark.sql.functions import lit, array
from pyspark.sql.types import ArrayType, FloatType

# Get all reviews
df_base = spark.table("group4.default.ai_reviews")
reviews_pdf = df_base.toPandas()

# Generate embeddings in batches
batch_size = 1000
all_embeddings = []

print(f"Processing {len(reviews_pdf)} reviews in batches of {batch_size}...")
for i in range(0, len(reviews_pdf), batch_size):
    batch = reviews_pdf["Review_Text"][i:i+batch_size].tolist()
    batch_embeddings = model.encode(batch, show_progress_bar=False)
    all_embeddings.extend(batch_embeddings.tolist())
    if (i // batch_size + 1) % 10 == 0:
        print(f"Processed {i + len(batch)} reviews...")

print("Embeddings generated. Creating DataFrame...")

# Add embeddings to the pandas DataFrame
reviews_pdf["embedding"] = all_embeddings

# Convert back to Spark DataFrame
df_with_embeddings = spark.createDataFrame(reviews_pdf)

# Write to table
print("Writing to Delta table...")
df_with_embeddings.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("group4.default.ai_reviews_embeddings")

print("✅ Table group4.default.ai_reviews_embeddings created successfully!")

# 3. Tooling Setup

### 3.1 Vector search / RAG tool

In [0]:
# RAG Vector Search Tool
# This function performs semantic similarity search and retrieves relevant reviews
# to provide context for the LLM agent's responses

from pyspark.sql.functions import col, udf, lit, struct
from pyspark.sql.types import DoubleType
import numpy as np
from sentence_transformers import SentenceTransformer
import pandas as pd

# Define cosine similarity UDF at module level (best practice for Spark)
def cosine_similarity(vec1, vec2):
    """Calculate cosine similarity between two vectors"""
    vec1_arr = np.array(vec1)
    vec2_arr = np.array(vec2)
    dot_product = np.dot(vec1_arr, vec2_arr)
    norm1 = np.linalg.norm(vec1_arr)
    norm2 = np.linalg.norm(vec2_arr)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return float(dot_product / (norm1 * norm2))

cosine_sim_udf = udf(cosine_similarity, DoubleType())

# Model will be loaded on first use to avoid environment sync issues
embedding_model = None

def get_embedding_model():
    """Lazy load the embedding model to avoid UDF serialization issues"""
    global embedding_model
    if embedding_model is None:
        print("Loading SentenceTransformer model...")
        embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
        print("✅ Model loaded successfully")
    return embedding_model

def vector_search(query: str, top_k: int = 10, min_similarity: float = 0.0):
    """
    Perform semantic similarity search on review embeddings.
    
    This is the core RAG retrieval function that:
    1. Converts the query to an embedding vector
    2. Computes cosine similarity with all review embeddings
    3. Returns the top-k most similar reviews
    
    Args:
        query (str): Natural language search query
        top_k (int): Number of top results to return (default: 10)
        min_similarity (float): Minimum similarity threshold 0-1 (default: 0.0)
    
    Returns:
        pd.DataFrame: Top-k most similar reviews with columns:
            - App: Application name
            - Review_Text: Full review text
            - Star_Rating: User rating (1-5)
            - Review_Date: Date of review
            - Sentiment_Polarity: Sentiment score (-1 to 1)
            - Review_Theme: Categorized theme
            - similarity_score: Cosine similarity (0-1)
    """
    # Generate query embedding
    model = get_embedding_model()
    query_embedding = model.encode(query).tolist()
    
    # Load embeddings table
    embeddings_df = spark.table("group4.default.ai_reviews_embeddings")
    
    # Compute similarity scores
    results_df = embeddings_df.withColumn(
        "similarity_score",
        cosine_sim_udf(col("embedding"), lit(query_embedding))
    )
    
    # Filter by minimum similarity and get top_k
    if min_similarity > 0:
        results_df = results_df.filter(col("similarity_score") >= min_similarity)
    
    results_df = results_df.orderBy(col("similarity_score").desc()).limit(top_k)
    
    # Select relevant columns and convert to pandas for agent consumption
    return results_df.select(
        "App",
        "Review_Text",
        "Star_Rating",
        "Review_Date",
        "Sentiment_Polarity",
        "Review_Theme",
        "similarity_score"
    ).toPandas()

def format_context_for_llm(results_df: pd.DataFrame) -> str:
    """
    Format retrieved reviews into a context string for the LLM.
    
    Args:
        results_df (pd.DataFrame): Results from vector_search()
    
    Returns:
        str: Formatted context string with review information
    """
    if len(results_df) == 0:
        return "No relevant reviews found."
    
    context_parts = []
    for idx, row in results_df.iterrows():
        context_parts.append(
            f"Review {idx+1} (Similarity: {row['similarity_score']:.3f}):\n"
            f"App: {row['App']}\n"
            f"Rating: {row['Star_Rating']}/5\n"
            f"Date: {row['Review_Date']}\n"
            f"Theme: {row['Review_Theme']}\n"
            f"Sentiment: {row['Sentiment_Polarity']:.2f}\n"
            f"Review: {row['Review_Text']}\n"
        )
    
    return "\n---\n\n".join(context_parts)

# Test the vector search function
print("\n=== Testing Vector Search Tool ===")
test_query = "What do users complain about app crashes?"
print(f"Query: {test_query}\n")

test_results = vector_search(test_query, top_k=5)
print(f"Found {len(test_results)} similar reviews:\n")
display(test_results[["App", "Star_Rating", "similarity_score", "Review_Theme", "Review_Text"]])

# Show formatted context for LLM
print("\n=== Formatted Context for LLM ===")
context = format_context_for_llm(test_results)
print(context[:1000] + "..." if len(context) > 1000 else context)

### 3.2 Custom functions

In [0]:
%sql
CREATE OR REPLACE FUNCTION group4.default.all_ai_applications_rating()
RETURNS TABLE
LANGUAGE SQL
COMMENT 'Shows all AI applications reveiw count, thumbs up count and average ratings'
RETURN (
  SELECT 
    `App`,
    count(*) as review_count,
    sum(`Thumbs_Up_Count`) as thumbs_up_count,
    avg(`Star_Rating`) as avg_rating
  FROM group4.default.ai_reviews
  GROUP BY 1
  ORDER BY 3 desc
)

In [0]:
%sql
select * from group4.default.all_ai_applications_rating()

In [0]:
%sql
CREATE OR REPLACE FUNCTION group4.default.fetch_review_theme_category()
RETURNS TABLE
LANGUAGE SQL
COMMENT 'Fetch review theme category'
RETURN (
  SELECT 
    `Review_Theme`,
    count(*) as review_count
  FROM group4.default.ai_reviews
  GROUP BY 1
  ORDER BY 2 desc
)

In [0]:
%sql
select * from group4.default.fetch_review_theme_category()

In [0]:
%sql
CREATE OR REPLACE FUNCTION group4.default.fetch_review_contents(
    theme_param STRING COMMENT 'theme category, e.g. Bugs/Performance',
    keyword_param STRING DEFAULT NULL COMMENT 'optional keyword to search in review text'
)
RETURNS TABLE
LANGUAGE SQL
COMMENT 'Fetch review contents by theme, with optional review text keyword'
RETURN (
    SELECT
        `App`,
        `Star_Rating`,
        `Review_Text`,
        `Word_Count`,
        `Review_Length_Chars`,
        `Thumbs_Up_Count`,
        `App_Version`,
        `Sentiment_Polarity`,
        `Review_Theme`
    FROM group4.default.ai_reviews
    WHERE `Review_Theme` = theme_param
      AND (
          keyword_param IS NULL
          OR lower(`Review_Text`) LIKE concat('%', lower(keyword_param), '%')
      )
);

In [0]:
%sql
select * 
from group4.default.fetch_review_contents('Bugs/Performance', 'crash') limit 10

In [0]:
%sql
CREATE OR REPLACE FUNCTION group4.default.find_most_used_words_by_model(
    theme_param STRING DEFAULT NULL COMMENT 'theme category (not used, for agent compatibility)',
    keyword_param STRING DEFAULT NULL COMMENT 'optional keyword (not used, for agent compatibility)'
)
RETURNS TABLE
LANGUAGE SQL
COMMENT 'Finds the 10 most frequently used keywords in the review texts from ai_reviews_embeddings table, grouped by model.'
RETURN (
  SELECT 
    `App` as model,
    lower(trim(word)) as keyword,
    count(*) as word_count
  FROM (
    SELECT `App`, explode(split(regexp_replace(`Review_Text`, '[^a-zA-Z\\s]', ' '), '\\s+')) as word
    FROM group4.default.ai_reviews_embeddings
  )
  WHERE length(trim(word)) > 2  -- Filter out very short words
  GROUP BY `App`, lower(trim(word))
  ORDER BY model, word_count DESC
)

In [0]:
%sql
select * from group4.default.find_most_used_words_by_model() limit 10

### 3.3 MCP tools if used

In [0]:
from typing import Optional, List, Dict, Any, Callable
import os

# MCP-Style External Tools Manager
# Note: Native MCP connectors (Google Drive, Slack, GitHub, etc.) are only available
# in the Databricks Assistant UI and cannot be accessed programmatically from notebooks.
# This class provides a pattern for adding external API tools to your custom agent.

class ExternalToolsManager:
    """
    Manages external tools integration for the custom agent.
    
    While Databricks MCP connectors are not accessible from custom code,
    you can add external services by:
    1. Direct API calls (shown below)
    2. Unity Catalog external functions
    3. Custom Python functions that wrap external APIs
    """
    
    def __init__(self):
        self.external_tools: List[Dict[str, Any]] = []
    
    def register_external_tool(self, 
                               tool_name: str,
                               description: str,
                               parameters: Dict[str, Any],
                               exec_function: Callable) -> Dict[str, Any]:
        """
        Register an external tool with OpenAI function-calling spec.
        
        Args:
            tool_name: Unique name for the tool
            description: What the tool does
            parameters: OpenAI function parameters schema
            exec_function: Callable that executes the tool
        
        Returns:
            Tool specification dict
        """
        tool_spec = {
            "type": "function",
            "function": {
                "name": tool_name,
                "description": description,
                "parameters": parameters
            }
        }
        
        self.external_tools.append({
            "spec": tool_spec,
            "exec_fn": exec_function
        })
        
        return tool_spec
    
    def get_tool_infos_for_agent(self) -> List[Dict[str, Any]]:
        """
        Get tool specifications in ToolInfo format for the agent.
        
        Returns:
            List of tool info dicts compatible with react_agent.py
        """
        return self.external_tools


# ============================================================================
# Example: Web Search Tool (you can add API key and implement actual search)
# ============================================================================

def web_search_tool(query: str, num_results: int = 5) -> str:
    """
    Real web search implementation with Serper API (primary) and DuckDuckGo (fallback).
    
    Priority:
    1. Serper API (if API key is configured in Databricks Secrets)
    2. DuckDuckGo (fallback, no API key required)
    
    Args:
        query: Search query
        num_results: Number of results to return
    
    Returns:
        Formatted search results as markdown
    """
    import requests
    
    # Try Serper API first (if configured)
    try:
        api_key = dbutils.secrets.get(scope="default", key="serper-api-key")
        if api_key and api_key.strip():
            response = requests.post(
                "https://google.serper.dev/search",
                json={"q": query, "num": num_results},
                headers={"X-API-KEY": api_key, "Content-Type": "application/json"},
                timeout=10
            )
            
            if response.status_code == 200:
                data = response.json()
                results = data.get("organic", [])
                
                if not results:
                    return f"No results found for '{query}'."
                
                formatted = f"Web search results for '{query}':\n\n"
                for i, result in enumerate(results[:num_results], 1):
                    title = result.get("title", "No title")
                    url = result.get("link", "")
                    snippet = result.get("snippet", "No description")
                    formatted += f"{i}. **{title}**\n   URL: {url}\n   Snippet: {snippet}\n\n"
                
                return formatted
    except Exception as e:
        # Serper failed or not configured, fall back to DuckDuckGo
        pass
    
    # Fallback: DuckDuckGo (no API key required)
    try:
        from duckduckgo_search import DDGS
        
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=num_results))
        
        if not results:
            return f"No results found for '{query}'."
        
        formatted = f"Web search results for '{query}':\n\n"
        for i, result in enumerate(results, 1):
            title = result.get("title", "No title")
            url = result.get("href", "")
            snippet = result.get("body", "No description")
            formatted += f"{i}. **{title}**\n   URL: {url}\n   Snippet: {snippet}\n\n"
        
        return formatted
        
    except Exception as e:
        return f"Web search failed: {str(e)}\n\nPlease ensure duckduckgo-search is installed: %pip install duckduckgo-search"


# ============================================================================
# Example: Document Retrieval Tool (using Databricks UC Volume)
# ============================================================================

def document_search_tool(query: str, top_k: int = 5) -> str:
    """
    Semantic document search using vector embeddings.
    
    Searches the AI reviews embeddings table using cosine similarity.
    Uses BAAI/bge-small-en-v1.5 for query encoding.
    
    Args:
        query: Search query
        top_k: Number of most similar documents to return
    
    Returns:
        Formatted relevant reviews with similarity scores
    """
    try:
        from sentence_transformers import SentenceTransformer
        import numpy as np
        from pyspark.sql import functions as F
        
        # Load embedding model
        model = SentenceTransformer('BAAI/bge-small-en-v1.5')
        
        # Encode query
        query_embedding = model.encode(query, normalize_embeddings=True)
        
        # Load embeddings table
        embeddings_df = spark.table("group4.default.ai_application_reviews_with_embeddings")
        
        # Convert to pandas for similarity computation
        df_pandas = embeddings_df.select(
            "Application_Name", "Review", "Rating", "Review_Date",
            "Review_Theme", "Sentiment_Score", "embedding"
        ).toPandas()
        
        # Compute cosine similarities
        similarities = []
        for idx, row in df_pandas.iterrows():
            doc_embedding = np.array(row['embedding'])
            similarity = np.dot(query_embedding, doc_embedding)
            similarities.append((similarity, row))
        
        # Sort by similarity and get top K
        similarities.sort(reverse=True, key=lambda x: x[0])
        top_results = similarities[:top_k]
        
        # Format results
        formatted = f"Found {len(top_results)} relevant reviews for '{query}':\n\n"
        for i, (sim_score, row) in enumerate(top_results, 1):
            formatted += f"\n{i}. **{row['Application_Name']}** (Similarity: {sim_score:.3f})\n"
            formatted += f"   Rating: {row['Rating']}/5 | Date: {row['Review_Date']}\n"
            formatted += f"   Theme: {row['Review_Theme']} | Sentiment: {row['Sentiment_Score']:.2f}\n"
            formatted += f"   Review: {row['Review'][:200]}{'...' if len(str(row['Review'])) > 200 else ''}\n"
        
        return formatted
        
    except Exception as e:
        return f"Document search failed: {str(e)}\n\nPlease ensure sentence-transformers is installed: %pip install sentence-transformers"


# ============================================================================
# Initialize and Register Tools
# ============================================================================

external_tools_manager = ExternalToolsManager()

# Register web search tool
web_search_spec = external_tools_manager.register_external_tool(
    tool_name="web_search",
    description="Search the web for current information, news, or facts not in the review database",
    parameters={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query"
            },
            "num_results": {
                "type": "integer",
                "description": "Number of results to return (default: 5)",
                "default": 5
            }
        },
        "required": ["query"]
    },
    exec_function=web_search_tool
)

# Register document search tool
doc_search_spec = external_tools_manager.register_external_tool(
    tool_name="document_search",
    description="Search internal documents and knowledge base for relevant information",
    parameters={
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query"
            },
            "top_k": {
                "type": "integer",
                "description": "Number of documents to return (default: 3)",
                "default": 3
            }
        },
        "required": ["query"]
    },
    exec_function=document_search_tool
)

print("✅ External tools manager initialized")
print(f"📦 Registered {len(external_tools_manager.external_tools)} external tools:")
for tool in external_tools_manager.external_tools:
    print(f"   - {tool['spec']['function']['name']}: {tool['spec']['function']['description']}")

print("\n💡 To add these tools to your agent:")
print("   1. Import ToolInfo from react_agent.py")
print("   2. For each external tool, create a ToolInfo object")
print("   3. Add to TOOL_INFOS list before agent initialization")
print("\n⚠️  Note: Web search and document search are placeholder implementations.")
print("   Add your actual API keys and logic to make them functional.")

In [0]:
# Bridge: Convert External Tools to Agent-Compatible Format
# This cell creates ToolInfo-compatible objects that can be imported by the agent

from typing import List, Dict, Any

class ExternalToolInfo:
    """
    Simplified ToolInfo structure for external tools.
    Will be converted to actual ToolInfo in react_agent.py
    """
    def __init__(self, name: str, spec: dict, exec_fn):
        self.name = name
        self.spec = spec
        self.exec_fn = exec_fn

# Convert registered external tools to ToolInfo format
EXTERNAL_TOOL_INFOS: List[ExternalToolInfo] = []

for tool_data in external_tools_manager.external_tools:
    tool_info = ExternalToolInfo(
        name=tool_data["spec"]["function"]["name"],
        spec=tool_data["spec"],
        exec_fn=tool_data["exec_fn"]
    )
    EXTERNAL_TOOL_INFOS.append(tool_info)

print(f"\n✅ Converted {len(EXTERNAL_TOOL_INFOS)} external tools to agent-compatible format")
print("\n📋 External tools ready for agent:")
for tool_info in EXTERNAL_TOOL_INFOS:
    print(f"   - {tool_info.name}")

print("\n💡 These tools will be automatically added to the agent in the next cell.")

# 4. Agent Definition

### 4.1 Tools available to the agent

### 4.2 Agent workflow / graph

In [0]:
%pip install backoff
#dbutils.library.restartPython()

In [0]:
%%writefile react_agent.py
import copy
import json
import warnings
from typing import Any, Callable, Generator, Optional
from uuid import uuid4

import backoff
import mlflow
import numpy as np
import openai
import pandas as pd
from databricks.sdk import WorkspaceClient
from mlflow.entities import SpanType
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
    to_chat_completions_input,
)
from openai import OpenAI
from pydantic import BaseModel
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, udf
from pyspark.sql.types import DoubleType
from sentence_transformers import SentenceTransformer
from databricks_openai import UCFunctionToolkit
from unitycatalog.ai.core.base import get_uc_function_client


############################################
# LLM endpoint — replaced by notebook cell
############################################
LLM_ENDPOINT_NAME = "__LLM_ENDPOINT__"

SYSTEM_PROMPT = """You are an AI Review Analysis Assistant specializing in generative AI application reviews.

Your role is to help users understand user feedback and sentiment about AI applications like
ChatGPT, Claude, Gemini, Midjourney, and other generative AI tools.

You have access to one tool:
- group4__default__all_ai_applications_rating: list all AI applications and their average rating.
- group4__default__fetch_review_theme_category: fetch all review theme categories.
- group4__default__fetch_review_contents: retrieve review content for a specific category, with an optional keyword filter, while limiting the number of records returned per request.
- group4__default__find_most_used_words_by_model: Finds the 10 most frequently used keywords in the review texts from ai_reviews_embeddings table, grouped by model.


For every user question, follow this process:
1. Think about what information is needed.
2. Choose the correct tool.
3. Review the tool result.
4. If more information is needed, call another tool.
5. Then provide the final answer.

Always ground the final answer in tool results."""


############################################
# Embedding model — lazy loaded once
############################################
_embedding_model: Optional[SentenceTransformer] = None

def _get_embedding_model() -> SentenceTransformer:
    global _embedding_model
    if _embedding_model is None:
        _embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
    return _embedding_model

############################################
# Cosine similarity Spark UDF
############################################
def _cosine_similarity(vec1, vec2) -> float:
    a = np.array(vec1)
    b = np.array(vec2)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom != 0 else 0.0

_cosine_sim_udf = udf(_cosine_similarity, DoubleType())

############################################
# Core retrieval logic
############################################
def _vector_search_exec(
    query: str,
    top_k: int = 10,
    min_similarity: float = 0.0,
) -> str:
    """
    Encode query, compute cosine similarity against ai_reviews_embeddings,
    return formatted context string ready for the LLM.
    """
    spark = SparkSession.builder.getOrCreate()
    model = _get_embedding_model()
    query_vec = model.encode(query).tolist()

    df = spark.table("group4.default.ai_reviews_embeddings")
    df = df.withColumn(
        "similarity_score",
        _cosine_sim_udf(col("embedding"), lit(query_vec)),
    )

    if min_similarity > 0:
        df = df.filter(col("similarity_score") >= min_similarity)

    results: pd.DataFrame = (
        df.orderBy(col("similarity_score").desc())
        .limit(int(top_k))
        .select(
            "App",
            "Review_Text",
            "Star_Rating",
            "Review_Date",
            "Sentiment_Polarity",
            "Review_Theme",
            "similarity_score",
        )
        .toPandas()
    )

    if results.empty:
        return "No relevant reviews found."

    parts = []
    for i, row in results.iterrows():
        parts.append(
            f"Review {i + 1} (Similarity: {row['similarity_score']:.3f}):\n"
            f"App: {row['App']}\n"
            f"Rating: {row['Star_Rating']}/5\n"
            f"Date: {row['Review_Date']}\n"
            f"Theme: {row['Review_Theme']}\n"
            f"Sentiment: {row['Sentiment_Polarity']:.2f}\n"
            f"Review: {row['Review_Text']}\n"
        )
    return "\n---\n\n".join(parts)

############################################
# Tool specs (OpenAI function-calling format)
############################################
_VECTOR_SEARCH_SPEC = {
    "type": "function",
    "function": {
        "name": "vector_search",
        "description": (
            "Search reviews for qualitative topics: complaints, praise, specific user experiences. "
            "Do NOT use for aggregate statistics, rankings, or average ratings — "
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Natural language search query describing what reviews to find",
                },
                "top_k": {
                    "type": "integer",
                    "description": "Number of top results to return (default: 10)",
                    "default": 10,
                },
                "min_similarity": {
                    "type": "number",
                    "description": "Minimum similarity threshold 0-1 (default: 0.0)",
                    "default": 0.0,
                },
            },
            "required": ["query"],
        },
    },
}

############################################
# ToolInfo + agent boilerplate (unchanged)
############################################
class ToolInfo(BaseModel):
    name: str
    spec: dict
    exec_fn: Callable

    class Config:
        arbitrary_types_allowed = True

def create_tool_info(tool_spec, exec_fn_param: Optional[Callable] = None):
    tool_spec["function"].pop("strict", None)
    tool_name = tool_spec["function"]["name"]
    udf_name = tool_name.replace("__", ".")

    # Define a wrapper that accepts kwargs for the UC tool call,
    # then passes them to the UC tool execution client
    def exec_fn(**kwargs):
        function_result = uc_function_client.execute_function(udf_name, kwargs)
        if function_result.error is not None:
            return function_result.error
        else:
            return function_result.value
    return ToolInfo(name=tool_name, spec=tool_spec, exec_fn=exec_fn_param or exec_fn)


def _sanitize_tool_spec(spec: dict) -> dict:
    """Remove JSON schema keywords that some model endpoints reject."""
    spec = copy.deepcopy(spec)
    params = spec.get("function", {}).get("parameters") or {}
    if not isinstance(params, dict) or "properties" not in params:
        return spec
    for prop in params.get("properties", {}).values():
        if not isinstance(prop, dict):
            continue
        t = prop.get("type")
        if t == "string":
            for key in ("minLength", "maxLength", "pattern", "format"):
                prop.pop(key, None)
        elif t in ("integer", "number"):
            for key in ("minimum", "maximum", "exclusiveMinimum", "exclusiveMaximum", "format"):
                prop.pop(key, None)
        elif t == "array":
            for key in ("minItems", "maxItems", "uniqueItems"):
                prop.pop(key, None)
            items = prop.get("items")
            if isinstance(items, dict):
                for key in ("minLength", "maxLength", "pattern", "format",
                            "minimum", "maximum", "exclusiveMinimum", "exclusiveMaximum"):
                    items.pop(key, None)
    return spec


# ---- Register tools here ----
TOOL_INFOS: list[ToolInfo] = [
    # disable vetor search, it is so slow
    # ToolInfo(
    #     name="vector_search",
    #     spec=_VECTOR_SEARCH_SPEC,
    #     exec_fn=_vector_search_exec,
    # ),
]

# Add Unity Catalog functions
UC_TOOL_NAMES = ["group4.default.all_ai_applications_rating", 
                 "group4.default.fetch_review_theme_category", 
                 "group4.default.fetch_review_contents",
                 "group4.default.find_most_used_words_by_model"
                 ]

uc_toolkit = UCFunctionToolkit(function_names=UC_TOOL_NAMES)
uc_function_client = get_uc_function_client()
for tool_spec in uc_toolkit.tools:
    TOOL_INFOS.append(create_tool_info(tool_spec))

############################################
# ToolCallingAgent (structure unchanged)
############################################
class ToolCallingAgent(ResponsesAgent):
    def __init__(self, llm_endpoint: str, tools: list[ToolInfo]):
        self.llm_endpoint = llm_endpoint
        self.workspace_client = WorkspaceClient()
        self.model_serving_client: OpenAI = (
            self.workspace_client.serving_endpoints.get_open_ai_client()
        )
        self._tools_dict = {tool.name: tool for tool in tools}

    def get_tool_specs(self) -> list[dict]:
        return [_sanitize_tool_spec(t.spec) for t in self._tools_dict.values()]

    @mlflow.trace(span_type=SpanType.TOOL)
    def execute_tool(self, tool_name: str, args: dict) -> Any:
        sane_args = {k: v for k, v in (args or {}).items() if k and isinstance(k, str)}
        name = tool_name.strip().strip('"').strip("'")
        if "<" in name:
            name = name.split("<")[0].strip()
        if name in self._tools_dict:
            return self._tools_dict[name].exec_fn(**sane_args)
        candidates = [k for k in self._tools_dict if name.startswith(k)]
        if candidates:
            return self._tools_dict[max(candidates, key=len)].exec_fn(**sane_args)
        raise KeyError(f"Unknown tool: {tool_name!r}. Known tools: {list(self._tools_dict.keys())}")

    def call_llm(self, messages: list[dict[str, Any]]) -> Generator[dict[str, Any], None, None]:
        for chunk in self.model_serving_client.chat.completions.create(
            model=self.llm_endpoint,
            messages=to_chat_completions_input(messages),
            tools=self.get_tool_specs(),
            stream=True,
        ):
            chunk_dict = chunk.to_dict()
            if chunk_dict.get("choices"):
                yield chunk_dict

    def handle_tool_call(
        self,
        tool_call: dict[str, Any],
        messages: list[dict[str, Any]],
    ) -> ResponsesAgentStreamEvent:
        try:
            args = json.loads(tool_call.get("arguments"))
        except Exception:
            args = {}
        result = str(self.execute_tool(tool_name=tool_call["name"], args=args))
        tool_call_output = self.create_function_call_output_item(tool_call["call_id"], result)
        messages.append(tool_call_output)
        return ResponsesAgentStreamEvent(type="response.output_item.done", item=tool_call_output)

    def call_and_run_tools(
        self,
        messages: list[dict[str, Any]],
        max_iter: int = 20,
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        for _ in range(max_iter):
            last_msg = messages[-1]
            if last_msg.get("role") == "assistant":
                return
            elif last_msg.get("type") == "function_call":
                yield self.handle_tool_call(last_msg, messages)
            else:
                yield from output_to_responses_items_stream(
                    chunks=self.call_llm(messages), aggregator=messages
                )
        yield ResponsesAgentStreamEvent(
            type="response.output_item.done",
            item=self.create_text_output_item("Max iterations reached. Stopping.", str(uuid4())),
        )

    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)
            session_id = None
            if request.custom_inputs and "session_id" in request.custom_inputs:
                session_id = request.custom_inputs.get("session_id")
            elif request.context and request.context.conversation_id:
                session_id = request.context.conversation_id
            if session_id:
                mlflow.update_current_trace(metadata={"mlflow.trace.session": session_id})
            outputs = [
                event.item
                for event in self.predict_stream(request)
                if event.type == "response.output_item.done"
            ]
            return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)

    def predict_stream(
        self, request: ResponsesAgentRequest
    ) -> Generator[ResponsesAgentStreamEvent, None, None]:
        session_id = None
        if request.custom_inputs and "session_id" in request.custom_inputs:
            session_id = request.custom_inputs.get("session_id")
        elif request.context and request.context.conversation_id:
            session_id = request.context.conversation_id
        if session_id:
            mlflow.update_current_trace(metadata={"mlflow.trace.session": session_id})
        messages = to_chat_completions_input([i.model_dump() for i in request.input])
        if SYSTEM_PROMPT:
            messages.insert(0, {"role": "system", "content": SYSTEM_PROMPT})
        yield from self.call_and_run_tools(messages=messages)


mlflow.openai.autolog()

In [0]:
# Available foundation model endpoints (adjust for your workspace)
AVAILABLE_LLMS = [
    "databricks-gpt-oss-120b",
    "databricks-gpt-oss-20b",
    "databricks-llama-4-maverick",
]
# Set the LLM: by index (e.g. 0, 1, 2) or by name
CHOSEN_LLM_ENDPOINT = AVAILABLE_LLMS[0]  # change index or set CHOSEN_LLM_ENDPOINT = "your-endpoint-name"
print(f"Using LLM endpoint: {CHOSEN_LLM_ENDPOINT}")

In [0]:
# Bake chosen LLM into agent.py (run "Choose LLM" first)
with open("react_agent.py") as f:
    content = f.read()
with open("react_agent.py", "w") as f:
    f.write(content.replace("__LLM_ENDPOINT__", CHOSEN_LLM_ENDPOINT))
print("react_agent.py updated with LLM_ENDPOINT_NAME =", CHOSEN_LLM_ENDPOINT)

# 5. Agent Test Examples

### 5.1 Normal user queries

In [0]:
import traceback
import nest_asyncio
import sys
import os
import importlib.util

# Add the repo directory to Python path so react_agent.py can be imported
current_dir = "/Workspace/Repos/mzamaniturk@sandiego.edu/aai-510-finalproject-group-4"
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)
print(f"Added to Python path: {current_dir}")

# Use importlib to load the module directly - avoids import caching issues
module_path = os.path.join(current_dir, "react_agent.py")
spec = importlib.util.spec_from_file_location("react_agent", module_path)
react_agent = importlib.util.module_from_spec(spec)
sys.modules['react_agent'] = react_agent
spec.loader.exec_module(react_agent)
print("✅ Successfully loaded react_agent module")

nest_asyncio.apply()

# Add external tools (MCP-style) to the agent's tool list
if 'EXTERNAL_TOOL_INFOS' in globals() and EXTERNAL_TOOL_INFOS:
    print(f"\n✅ Adding {len(EXTERNAL_TOOL_INFOS)} external tools to agent...")
    for ext_tool in EXTERNAL_TOOL_INFOS:
        # Convert to react_agent.ToolInfo format
        tool_info = react_agent.ToolInfo(
            name=ext_tool.name,
            spec=ext_tool.spec,
            exec_fn=ext_tool.exec_fn
        )
        react_agent.TOOL_INFOS.append(tool_info)
    print(f"   Total tools available: {len(react_agent.TOOL_INFOS)}")
else:
    print("\nℹ️  No external tools found. Using UC functions only.")
    print(f"   Total tools available: {len(react_agent.TOOL_INFOS)}")

AGENT = react_agent.ToolCallingAgent(llm_endpoint=react_agent.LLM_ENDPOINT_NAME, tools=react_agent.TOOL_INFOS)
import mlflow
mlflow.models.set_model(AGENT)

In [0]:
react_agent.TOOL_INFOS

In [0]:
# Because the free version of Databricks does not support Vector Search indexes, we implemented similarity search using a custom embedding model, BAAI/bge-small-en-v1.5. For each query, the data is reloaded into Spark, so it may take mintues to compelete.

AGENT.predict(
    {"input": [{"role": "user", "content": "What do users complain about app crashes?"}], "custom_inputs": {"session_id": "test-session-123"}},
)

### 5.2 Tool calls

In [0]:
# this will use tool all_ai_applications_rating to reture all AI applications rating
AGENT.predict(
    {"input": [{"role": "user", "content": "List all AI applications and their average rating"}], "custom_inputs": {"session_id": "test-session-123"}},
)

In [0]:
%pip install duckduckgo-search

In [0]:
# Test the web_search MCP tool (currently a placeholder)
# This should trigger the agent to call the web_search tool

response = AGENT.predict(
    {
        "input": [{
            "role": "user",
            "content": "What are the latest AI trends in 2026? Search the web for recent developments."
        }],
        "custom_inputs": {"session_id": "test-mcp-websearch"}
    }
)

print("\n" + "="*60)
print("AGENT RESPONSE:")
print("="*60)
if isinstance(response, dict) and "choices" in response:
    print(response["choices"][0]["message"]["content"])
else:
    # Handle ResponsesAgentResponse object
    for item in response.output:
        if hasattr(item, 'type'):
            if item.type == 'message':
                print(item.content[0]['text'])
            elif item.type == 'function_call':
                print(f"\n🔧 Tool Called: {item.name}")
                print(f"   Arguments: {item.arguments}")
            elif item.type == 'function_call_output':
                print(f"\n📤 Tool Output: {item.output[:200]}..." if len(str(item.output)) > 200 else f"\n📤 Tool Output: {item.output}")

In [0]:
# Test the document_search MCP tool (currently a placeholder)
# This should trigger the agent to call the document_search tool

response = AGENT.predict(
    {
        "input": [{
            "role": "user",
            "content": "Search internal documents about AI safety guidelines and best practices."
        }],
        "custom_inputs": {"session_id": "test-mcp-docsearch"}
    }
)

print("\n" + "="*60)
print("AGENT RESPONSE:")
print("="*60)

for item in response.output:
    if hasattr(item, 'type'):
        if item.type == 'message':
            print(item.content[0]['text'])
        elif item.type == 'function_call':
            print(f"\n🔧 Tool Called: {item.name}")
            print(f"   Arguments: {item.arguments}")
        elif item.type == 'function_call_output':
            print(f"\n📤 Tool Output: {item.output[:200]}..." if len(str(item.output)) > 200 else f"\n📤 Tool Output: {item.output}")

In [0]:
# Display all available tools with detailed information
import pandas as pd

tools_info = []
for tool in react_agent.TOOL_INFOS:
    tool_spec = tool.spec['function']
    tools_info.append({
        'Tool Name': tool.name,
        'Description': tool_spec['description'],
        'Type': 'UC Function' if 'group4__default__' in tool.name else 'MCP External',
        'Parameters': ', '.join(tool_spec['parameters'].get('required', [])) or 'None',
        'Status': '✅ Functional' if ('group4__default__' in tool.name or tool.name in ['web_search', 'document_search']) else '⚠️ Placeholder'
    })

tools_df = pd.DataFrame(tools_info)

print("\n" + "="*80)
print("ALL AVAILABLE AGENT TOOLS")
print("="*80 + "\n")

display(tools_df)

print("\n" + "="*80)
print(f"TOTAL TOOLS: {len(tools_info)}")
print(f"  - UC Function Tools: {len([t for t in tools_info if t['Type'] == 'UC Function'])}")
print(f"  - MCP External Tools: {len([t for t in tools_info if t['Type'] == 'MCP External'])}")
print("="*80)

In [0]:
# Test the web_search MCP tool with REAL DuckDuckGo implementation
print("🧪 Testing REAL Web Search with DuckDuckGo...\n")

response = AGENT.predict(
    {
        "input": [{
            "role": "user",
            "content": "Search the web for the latest news about artificial intelligence regulation in 2026"
        }],
        "custom_inputs": {"session_id": "test-real-websearch"}
    }
)

print("\n" + "="*80)
print("AGENT RESPONSE WITH REAL WEB SEARCH")
print("="*80)

for item in response.output:
    if hasattr(item, 'type'):
        if item.type == 'message':
            print("\n📝 Final Answer:")
            print(item.content[0]['text'])
        elif item.type == 'function_call':
            print(f"\n🔧 Tool Called: {item.name}")
            print(f"   Arguments: {item.arguments}")
        elif item.type == 'function_call_output':
            output_str = str(item.output)
            if len(output_str) > 500:
                print(f"\n📤 Tool Output (first 500 chars):\n{output_str[:500]}...")
            else:
                print(f"\n📤 Tool Output:\n{output_str}")

print("\n" + "="*80)

In [0]:
# Test web search with a query that will return real web results
print("🌐 Testing Web Search with Current Topics...\n")

response = AGENT.predict(
    {
        "input": [{
            "role": "user",
            "content": "Search the web for information about ChatGPT capabilities and features"
        }],
        "custom_inputs": {"session_id": "test-real-topic"}
    }
)

print("\n" + "="*80)
print("AGENT RESPONSE - REAL WEB SEARCH RESULTS")
print("="*80)

for item in response.output:
    if hasattr(item, 'type'):
        if item.type == 'message':
            print("\n📝 Final Answer:")
            print(item.content[0]['text'])
        elif item.type == 'function_call':
            print(f"\n🔧 Tool Called: {item.name}")
        elif item.type == 'function_call_output':
            output_str = str(item.output)
            if len(output_str) > 800:
                print(f"\n📤 Tool Output (first 800 chars):")
                print(output_str[:800] + "...")
                print(f"\n   [Total output length: {len(output_str)} characters]")
            else:
                print(f"\n📤 Tool Output:\n{output_str}")

print("\n" + "="*80)

In [0]:
# Test web search with a query that will return real web results
print("🌐 Testing Web Search with Current Topics...\n")

response = AGENT.predict(
    {
        "input": [{
            "role": "user",
            "content": "Search the web for information about Gemini capabilities and features"
        }],
        "custom_inputs": {"session_id": "test-real-topic"}
    }
)

print("\n" + "="*80)
print("AGENT RESPONSE - REAL WEB SEARCH RESULTS")
print("="*80)

for item in response.output:
    if hasattr(item, 'type'):
        if item.type == 'message':
            print("\n📝 Final Answer:")
            print(item.content[0]['text'])
        elif item.type == 'function_call':
            print(f"\n🔧 Tool Called: {item.name}")
        elif item.type == 'function_call_output':
            output_str = str(item.output)
            if len(output_str) > 800:
                print(f"\n📤 Tool Output (first 800 chars):")
                print(output_str[:800] + "...")
                print(f"\n   [Total output length: {len(output_str)} characters]")
            else:
                print(f"\n📤 Tool Output:\n{output_str}")

print("\n" + "="*80)

### 5.3 Final answers

# 6. Evaluation & Traces

- At least 5 traces
- LLM judge or manual evaluation
- Success/failure analysis

In [0]:
# Evaluation Test Dataset
# Comprehensive test cases covering different query types and edge cases

import json
from typing import List, Dict, Any

test_dataset = [
    # ===== Factual Queries (Test UC Function Selection) =====
    {
        "query_id": "fact_001",
        "query": "List all AI applications and their average rating",
        "query_type": "factual",
        "expected_tools": ["group4__default__all_ai_applications_rating"],
        "expected_behavior": "Should call all_ai_applications_rating and return complete list",
        "eval_criteria": ["tool_selection", "correctness", "completeness"]
    },
    {
        "query_id": "fact_002",
        "query": "Which AI application has the highest rating?",
        "query_type": "factual",
        "expected_tools": ["group4__default__all_ai_applications_rating"],
        "expected_behavior": "Should identify app with max avg_rating",
        "eval_criteria": ["tool_selection", "correctness", "groundedness"]
    },
    {
        "query_id": "fact_003",
        "query": "What review themes are available in the dataset?",
        "query_type": "factual",
        "expected_tools": ["group4__default__fetch_review_theme_category"],
        "expected_behavior": "Should list all review theme categories",
        "eval_criteria": ["tool_selection", "correctness", "completeness"]
    },
    
    # ===== Theme-Based Queries (Multi-Step Reasoning) =====
    {
        "query_id": "theme_001",
        "query": "What do users complain about regarding bugs and performance?",
        "query_type": "thematic",
        "expected_tools": ["group4__default__fetch_review_contents"],
        "expected_theme": "Bugs/Performance",
        "expected_behavior": "Should fetch Bugs/Performance reviews and synthesize complaints",
        "eval_criteria": ["tool_selection", "groundedness", "relevance", "synthesis"]
    },
    {
        "query_id": "theme_002",
        "query": "Show me reviews about app crashes",
        "query_type": "thematic",
        "expected_tools": ["group4__default__fetch_review_contents"],
        "expected_keyword": "crash",
        "expected_behavior": "Should use keyword filter for 'crash'",
        "eval_criteria": ["tool_selection", "parameter_usage", "groundedness"]
    },
    {
        "query_id": "theme_003",
        "query": "What features do users love about these AI apps?",
        "query_type": "thematic",
        "expected_tools": ["group4__default__fetch_review_contents"],
        "expected_theme": "Features",
        "expected_behavior": "Should fetch positive reviews about features",
        "eval_criteria": ["tool_selection", "groundedness", "sentiment_awareness"]
    },
    
    # ===== Comparison Queries =====
    {
        "query_id": "compare_001",
        "query": "Compare the ratings of ChatGPT and Claude",
        "query_type": "comparison",
        "expected_tools": ["group4__default__all_ai_applications_rating"],
        "expected_behavior": "Should extract and compare ratings for both apps",
        "eval_criteria": ["tool_selection", "correctness", "completeness"]
    },
    
    # ===== Out-of-Scope Queries (Should Reject Gracefully) =====
    {
        "query_id": "reject_001",
        "query": "What's the weather like today?",
        "query_type": "out_of_scope",
        "expected_tools": [],
        "expected_behavior": "Should politely decline and explain scope limitation",
        "eval_criteria": ["rejection_handling", "politeness", "scope_explanation"]
    },
    {
        "query_id": "reject_002",
        "query": "Write me Python code to scrape websites",
        "query_type": "out_of_scope",
        "expected_tools": [],
        "expected_behavior": "Should decline and redirect to review analysis capabilities",
        "eval_criteria": ["rejection_handling", "safety", "redirection"]
    },
]

print(f"✅ Test dataset created with {len(test_dataset)} test cases")
print(f"\nBreakdown by type:")
for qtype in ["factual", "thematic", "comparison", "out_of_scope"]:
    count = len([t for t in test_dataset if t["query_type"] == qtype])
    print(f"  - {qtype}: {count} queries")

print("\n📝 Sample queries:")
for test_case in test_dataset[:3]:
    print(f"  [{test_case['query_id']}] {test_case['query']}")

In [0]:
# LLM-as-Judge Evaluator
# Uses an LLM to evaluate agent responses across multiple dimensions

import openai
import json
import os
from openai import OpenAI
import react_agent

class LLMJudge:
    """LLM-as-judge evaluator for agent responses"""
    
    def __init__(self, llm_endpoint: str):
        # Get Databricks token from context
        try:
            token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
        except:
            # Fallback to environment variable if dbutils not available
            token = os.environ.get("DATABRICKS_TOKEN", "dapi000000000000000000000000000000")
        
        self.client = OpenAI(
            api_key=token,
            base_url=f"https://{spark.conf.get('spark.databricks.workspaceUrl')}/serving-endpoints"
        )
        self.llm_endpoint = llm_endpoint
    
    def _robust_parse_json(self, content, default_keys: dict) -> dict:
        """Robust JSON parser with multiple fallback strategies
        
        Args:
            content: Can be str, list, or dict
            default_keys: Default values to return if parsing fails
        """
        import re
        import ast
        
        # Strategy 0a: If content is already a list/dict object, handle directly
        if isinstance(content, list):
            # Format: [{'type': 'reasoning', ...}, {'type': 'text', 'text': '{...JSON...}'}]
            for item in content:
                if isinstance(item, dict) and item.get('type') == 'text':
                    json_text = item.get('text', '')
                    try:
                        return json.loads(json_text)
                    except:
                        pass
        
        # Strategy 0b: Handle nested structure from string representation
        if isinstance(content, str):
            try:
                # Try to parse as Python literal (handles the list/dict structure)
                parsed = ast.literal_eval(content)
                if isinstance(parsed, list):
                    # Find the item with 'type': 'text' and extract its 'text' field
                    for item in parsed:
                        if isinstance(item, dict) and item.get('type') == 'text':
                            json_text = item.get('text', '')
                            return json.loads(json_text)
            except:
                pass
        
        # Strategy 1: Direct JSON parse
        try:
            return json.loads(content)
        except:
            pass
        
        # Strategy 2: Extract JSON from markdown code blocks
        try:
            json_match = re.search(r'```(?:json)?\s*({.*?})\s*```', content, re.DOTALL)
            if json_match:
                return json.loads(json_match.group(1))
        except:
            pass
        
        # Strategy 3: Find last { ... } block (most likely to be the final JSON)
        try:
            # Find all JSON-like blocks
            json_blocks = re.findall(r'{[^{}]*(?:{[^{}]*}[^{}]*)*}', content, re.DOTALL)
            # Try parsing from last to first (most recent is usually the answer)
            for block in reversed(json_blocks):
                try:
                    result = json.loads(block)
                    if 'score' in result:  # Verify it's the right JSON
                        return result
                except:
                    continue
        except:
            pass
        
        # Strategy 4: Extract score and reasoning with regex
        result = default_keys.copy()
        try:
            # Try to find score
            score_patterns = [
                r'["\']?score["\']?\s*:\s*(\d+)',
                r'Score:\s*(\d+)',
                r'(\d+)\s*/\s*5',
            ]
            for pattern in score_patterns:
                match = re.search(pattern, content, re.IGNORECASE)
                if match:
                    result['score'] = int(match.group(1))
                    break
            
            # Try to find reasoning
            reasoning_patterns = [
                r'["\']?reasoning["\']?\s*:\s*["\']([^"\']*)["\"]',
                r'Reasoning:\s*(.+?)(?:\n|$)',
                r'Explanation:\s*(.+?)(?:\n|$)',
            ]
            for pattern in reasoning_patterns:
                match = re.search(pattern, content, re.IGNORECASE | re.DOTALL)
                if match:
                    result['reasoning'] = match.group(1).strip()
                    break
            
            # If we found a score, consider it successful
            if result['score'] > 0:
                return result
        except:
            pass
        
        # All strategies failed - log and return default
        print(f"⚠️  Failed to parse judge response. Raw content:")
        print(f"{content[:500]}...")  # Print first 500 chars
        result['reasoning'] = f"Failed to parse judge response (tried multiple strategies)"
        return result
    
    def evaluate_correctness(self, query: str, tool_results: str, answer: str) -> Dict[str, Any]:
        """Evaluate if the answer is factually correct based on tool results"""
        prompt = f"""You are an expert evaluator. Assess if the agent's answer is factually correct based on the tool results.

User Question: {query}

Tool Results: {tool_results}

Agent's Answer: {answer}

Evaluate the correctness on a scale of 1-5:
5 = Completely correct, all facts match tool results
4 = Mostly correct with minor inaccuracies
3 = Partially correct, some facts wrong
2 = Mostly incorrect
1 = Completely incorrect or hallucinated

Respond EXACTLY in this JSON format (no additional text):
{{
  "score": <1-5>,
  "reasoning": "<brief explanation>",
  "errors": ["<list of any factual errors>"]
}}"""
        
        response = self.client.chat.completions.create(
            model=self.llm_endpoint,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=500
        )
        
        content = response.choices[0].message.content
        default = {"score": 0, "reasoning": "Failed to parse", "errors": []}
        return self._robust_parse_json(content, default)
    
    def evaluate_relevance(self, query: str, answer: str) -> Dict[str, Any]:
        """Evaluate if the answer is relevant to the user's question"""
        prompt = f"""You are an expert evaluator. Assess if the answer is relevant to the user's question.

User Question: {query}

Agent's Answer: {answer}

Evaluate relevance on a scale of 1-5:
5 = Perfectly addresses the question
4 = Mostly relevant, minor tangents
3 = Partially relevant
2 = Barely relevant
1 = Completely off-topic

Respond EXACTLY in this JSON format (no additional text):
{{
  "score": <1-5>,
  "reasoning": "<brief explanation>"
}}"""
        
        response = self.client.chat.completions.create(
            model=self.llm_endpoint,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=300
        )
        
        content = response.choices[0].message.content
        
        # Handle case where content is already a Python object (list/dict)
        if isinstance(content, (list, dict)):
            content = str(content)
        
        default = {"score": 0, "reasoning": "Failed to parse"}
        return self._robust_parse_json(content, default)
    
    def evaluate_groundedness(self, tool_results: str, answer: str) -> Dict[str, Any]:
        """Evaluate if the answer is grounded in tool results (no hallucination)"""
        prompt = f"""You are an expert evaluator. Assess if the answer is grounded in the provided tool results or contains hallucinated information.

Tool Results: {tool_results}

Agent's Answer: {answer}

Evaluate groundedness on a scale of 1-5:
5 = All claims are directly supported by tool results
4 = Mostly grounded, minor unsupported claims
3 = Some claims not in tool results
2 = Many unsupported claims
1 = Heavily hallucinated

Respond EXACTLY in this JSON format (no additional text):
{{
  "score": <1-5>,
  "reasoning": "<brief explanation>",
  "hallucinations": ["<list of any hallucinated claims>"]
}}"""
        
        response = self.client.chat.completions.create(
            model=self.llm_endpoint,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=400
        )
        
        content = response.choices[0].message.content
        default = {"score": 0, "reasoning": "Failed to parse", "hallucinations": []}
        return self._robust_parse_json(content, default)
    
    def evaluate_rejection(self, query: str, answer: str) -> Dict[str, Any]:
        """Evaluate if out-of-scope queries are rejected gracefully"""
        prompt = f"""You are an expert evaluator. Assess if the agent gracefully rejected this out-of-scope question.

User Question: {query}

Agent's Answer: {answer}

Evaluate the rejection quality on a scale of 1-5:
5 = Politely declined, explained scope, offered alternatives
4 = Politely declined with clear scope explanation
3 = Declined but could be more helpful
2 = Harsh or unhelpful rejection
1 = Attempted to answer despite being out of scope

Respond EXACTLY in this JSON format (no additional text):
{{
  "score": <1-5>,
  "reasoning": "<brief explanation>",
  "was_rejected": <true/false>
}}"""
        
        response = self.client.chat.completions.create(
            model=self.llm_endpoint,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=300
        )
        
        content = response.choices[0].message.content
        default = {"score": 0, "reasoning": "Failed to parse", "was_rejected": False}
        return self._robust_parse_json(content, default)

# Initialize judge with a third LLM model (different from the agent)
# Agent uses: databricks-gpt-oss-120b
# Judge uses: databricks-meta-llama-3-3-70b-instruct (for unbiased evaluation)
JUDGE_LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
judge = LLMJudge(JUDGE_LLM_ENDPOINT)
print(f"✅ LLM Judge initialized with endpoint: {JUDGE_LLM_ENDPOINT}")
print(f"   Agent endpoint: {react_agent.LLM_ENDPOINT_NAME}")
print(f"   Judge endpoint: {JUDGE_LLM_ENDPOINT}")

In [0]:
# Run LLM-as-Judge Evaluation on Test Dataset
# Evaluates agent responses using the third LLM model as judge

import pandas as pd
import time
from typing import List, Dict, Any

def extract_tool_calls_and_output(response) -> tuple:
    """Extract tool calls and final answer from agent response"""
    tool_calls = []
    tool_outputs = []
    final_answer = ""
    
    for item in response.output:
        if hasattr(item, 'type'):
            if item.type == 'message':
                final_answer = item.content[0]['text'] if item.content else ""
            elif item.type == 'function_call':
                tool_calls.append({
                    'tool': item.name,
                    'arguments': item.arguments
                })
            elif item.type == 'function_call_output':
                tool_outputs.append(str(item.output))
    
    return tool_calls, tool_outputs, final_answer

def run_evaluation(test_cases: List[Dict], agent, judge) -> pd.DataFrame:
    """Run evaluation on test cases and return results dataframe"""
    results = []
    
    print(f"\n{'='*80}")
    print(f"RUNNING LLM-AS-JUDGE EVALUATION")
    print(f"{'='*80}\n")
    print(f"Test cases: {len(test_cases)}")
    print(f"Agent model: {react_agent.LLM_ENDPOINT_NAME}")
    print(f"Judge model: {JUDGE_LLM_ENDPOINT}\n")
    
    for i, test_case in enumerate(test_cases, 1):
        print(f"\n[{i}/{len(test_cases)}] Evaluating: {test_case['query_id']}")
        print(f"   Query: {test_case['query'][:80]}..." if len(test_case['query']) > 80 else f"   Query: {test_case['query']}")
        
        try:
            # Run agent
            start_time = time.time()
            response = agent.predict({
                "input": [{"role": "user", "content": test_case['query']}],
                "custom_inputs": {"session_id": f"eval-{test_case['query_id']}"}
            })
            response_time = time.time() - start_time
            
            # Extract components
            tool_calls, tool_outputs, final_answer = extract_tool_calls_and_output(response)
            tool_results_str = "\n\n".join(tool_outputs) if tool_outputs else "No tool calls"
            
            print(f"   Tools called: {[t['tool'] for t in tool_calls]}")
            print(f"   Response time: {response_time:.2f}s")
            
            # Evaluate based on query type
            eval_result = {
                'query_id': test_case['query_id'],
                'query': test_case['query'],
                'query_type': test_case['query_type'],
                'tools_called': [t['tool'] for t in tool_calls],
                'final_answer': final_answer,
                'response_time': response_time
            }
            
            if test_case['query_type'] == 'out_of_scope':
                # Evaluate rejection handling
                rejection_eval = judge.evaluate_rejection(test_case['query'], final_answer)
                eval_result.update({
                    'rejection_score': rejection_eval.get('score', 0),
                    'rejection_reasoning': rejection_eval.get('reasoning', ''),
                    'was_rejected': rejection_eval.get('was_rejected', False)
                })
                print(f"   Rejection score: {rejection_eval.get('score', 0)}/5")
            else:
                # Evaluate correctness, relevance, and groundedness
                correctness_eval = judge.evaluate_correctness(
                    test_case['query'], tool_results_str, final_answer
                )
                relevance_eval = judge.evaluate_relevance(
                    test_case['query'], final_answer
                )
                groundedness_eval = judge.evaluate_groundedness(
                    tool_results_str, final_answer
                )
                
                eval_result.update({
                    'correctness_score': correctness_eval.get('score', 0),
                    'correctness_reasoning': correctness_eval.get('reasoning', ''),
                    'relevance_score': relevance_eval.get('score', 0),
                    'relevance_reasoning': relevance_eval.get('reasoning', ''),
                    'groundedness_score': groundedness_eval.get('score', 0),
                    'groundedness_reasoning': groundedness_eval.get('reasoning', ''),
                    'avg_score': (correctness_eval.get('score', 0) + 
                                 relevance_eval.get('score', 0) + 
                                 groundedness_eval.get('score', 0)) / 3
                })
                
                print(f"   Scores - Correctness: {correctness_eval.get('score', 0)}/5, "
                      f"Relevance: {relevance_eval.get('score', 0)}/5, "
                      f"Groundedness: {groundedness_eval.get('score', 0)}/5")
            
            results.append(eval_result)
            
        except Exception as e:
            print(f"   ❌ Error: {str(e)}")
            results.append({
                'query_id': test_case['query_id'],
                'query': test_case['query'],
                'query_type': test_case['query_type'],
                'error': str(e)
            })
    
    return pd.DataFrame(results)

# Run the evaluation
print("Starting evaluation process...")
evaluation_results = run_evaluation(test_dataset, AGENT, judge)

print(f"\n{'='*80}")
print("EVALUATION COMPLETE")
print(f"{'='*80}\n")

In [0]:
# Display Evaluation Results Summary

import matplotlib.pyplot as plt
import numpy as np

print("\n" + "="*80)
print("EVALUATION RESULTS SUMMARY")
print("="*80 + "\n")

# Overall statistics
total_tests = len(evaluation_results)
errors = evaluation_results['error'].notna().sum() if 'error' in evaluation_results.columns else 0
successful = total_tests - errors

print(f"Total test cases: {total_tests}")
print(f"Successful: {successful}")
print(f"Errors: {errors}")
print()

# Breakdown by query type
print("\n📊 Results by Query Type:")
for qtype in evaluation_results['query_type'].unique():
    type_df = evaluation_results[evaluation_results['query_type'] == qtype]
    print(f"\n{qtype.upper()}:")
    print(f"  Count: {len(type_df)}")
    
    if qtype == 'out_of_scope':
        if 'rejection_score' in type_df.columns:
            avg_rejection = type_df['rejection_score'].mean()
            print(f"  Avg Rejection Score: {avg_rejection:.2f}/5")
            rejected_count = type_df['was_rejected'].sum() if 'was_rejected' in type_df.columns else 0
            print(f"  Properly Rejected: {rejected_count}/{len(type_df)}")
    else:
        if 'avg_score' in type_df.columns:
            avg_overall = type_df['avg_score'].mean()
            print(f"  Avg Overall Score: {avg_overall:.2f}/5")
            if 'correctness_score' in type_df.columns:
                avg_correctness = type_df['correctness_score'].mean()
                avg_relevance = type_df['relevance_score'].mean()
                avg_groundedness = type_df['groundedness_score'].mean()
                print(f"  Avg Correctness: {avg_correctness:.2f}/5")
                print(f"  Avg Relevance: {avg_relevance:.2f}/5")
                print(f"  Avg Groundedness: {avg_groundedness:.2f}/5")

# Display detailed results table
print("\n\n📋 DETAILED RESULTS:")
print("="*80 + "\n")

# Select relevant columns for display
if 'avg_score' in evaluation_results.columns:
    display_cols = ['query_id', 'query_type', 'correctness_score', 'relevance_score', 
                    'groundedness_score', 'avg_score']
    display_df = evaluation_results[display_cols].copy()
    # Convert numeric columns properly
    for col in ['correctness_score', 'relevance_score', 'groundedness_score', 'avg_score']:
        if col in display_df.columns:
            display_df[col] = pd.to_numeric(display_df[col], errors='coerce')
    print(display_df.to_string())
else:
    print(evaluation_results[['query_id', 'query', 'query_type', 'tools_called']].to_string())

# Visualization: Score distribution
if 'avg_score' in evaluation_results.columns:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    score_cols = ['correctness_score', 'relevance_score', 'groundedness_score']
    titles = ['Correctness', 'Relevance', 'Groundedness']
    
    for i, (col, title) in enumerate(zip(score_cols, titles)):
        if col in evaluation_results.columns:
            scores = evaluation_results[col].dropna()
            axes[i].hist(scores, bins=5, range=(1, 6), edgecolor='black', alpha=0.7)
            axes[i].set_xlabel('Score')
            axes[i].set_ylabel('Frequency')
            axes[i].set_title(f'{title} Score Distribution')
            axes[i].set_xticks([1, 2, 3, 4, 5])
            axes[i].axvline(scores.mean(), color='red', linestyle='--', 
                           label=f'Mean: {scores.mean():.2f}')
            axes[i].legend()
    
    plt.tight_layout()
    plt.show()

print("\n" + "="*80)
print("✅ Evaluation complete! Results saved in 'evaluation_results' DataFrame")
print("="*80)

In [0]:
# Sample Judge Reasoning Analysis
# Display detailed judge reasoning for selected queries

print("\n" + "="*80)
print("SAMPLE JUDGE REASONING ANALYSIS")
print("="*80 + "\n")

# Select interesting cases to examine
interesting_cases = [
    'fact_001',  # Low correctness despite tool use
    'fact_003',  # Perfect score
    'theme_003', # Failed thematic query
    'compare_001', # Perfect comparison
    'reject_001'  # Out-of-scope rejection
]

for query_id in interesting_cases:
    case = evaluation_results[evaluation_results['query_id'] == query_id].iloc[0]
    
    print(f"\n{'='*80}")
    print(f"Query ID: {query_id}")
    print(f"Query: {case['query']}")
    print(f"Type: {case['query_type']}")
    print(f"Tools Called: {case['tools_called']}")
    print(f"\nFinal Answer (first 200 chars):")
    print(f"{case['final_answer'][:200]}..." if len(str(case['final_answer'])) > 200 else case['final_answer'])
    print()
    
    if case['query_type'] == 'out_of_scope':
        print(f"\n📝 REJECTION EVALUATION:")
        print(f"Score: {case.get('rejection_score', 'N/A')}/5")
        print(f"Was Rejected: {case.get('was_rejected', 'N/A')}")
        print(f"Reasoning: {case.get('rejection_reasoning', 'N/A')}")
    else:
        print(f"\n📝 JUDGE EVALUATION:")
        print(f"Overall Score: {case.get('avg_score', 'N/A'):.2f}/5" if pd.notna(case.get('avg_score')) else "Overall Score: N/A")
        print()
        print(f"Correctness: {case.get('correctness_score', 'N/A')}/5")
        print(f"  Reasoning: {case.get('correctness_reasoning', 'N/A')}")
        print()
        print(f"Relevance: {case.get('relevance_score', 'N/A')}/5")
        print(f"  Reasoning: {case.get('relevance_reasoning', 'N/A')}")
        print()
        print(f"Groundedness: {case.get('groundedness_score', 'N/A')}/5")
        print(f"  Reasoning: {case.get('groundedness_reasoning', 'N/A')}")
    
    if 'error' in case and pd.notna(case['error']):
        print(f"\n⚠️ ERROR: {case['error']}")

print("\n" + "="*80)
print("KEY FINDINGS:")
print("="*80)
print()
print("✅ Strengths:")
print("  * Agent correctly rejects out-of-scope queries (weather)")
print("  * Perfect scores on simple factual queries (theme categories)")
print("  * Excellent comparison capability (ChatGPT vs Claude)")
print("  * All responses are well-grounded (no hallucination)")
print()
print("⚠️ Areas for Improvement:")
print("  * Some factual queries have relevance/correctness mismatch")
print("  * Thematic queries struggle with synthesis and relevance")
print("  * One out-of-scope query not properly rejected (code generation)")
print("  * One thematic query failed due to parameter error")
print()
print("🎯 Recommendations:")
print("  1. Improve thematic query handling and response synthesis")
print("  2. Strengthen out-of-scope detection for code generation requests")
print("  3. Fix parameter handling in fetch_review_contents function")
print("  4. Enhance relevance matching between query intent and final answer")
print()

In [0]:
# Save Evaluation Results to Unity Catalog
# Persist the evaluation results for future analysis and reporting

import datetime

print("\n" + "="*80)
print("SAVING EVALUATION RESULTS TO UNITY CATALOG")
print("="*80 + "\n")

# Add timestamp and metadata
evaluation_results['evaluation_timestamp'] = datetime.datetime.now()
evaluation_results['agent_model'] = react_agent.LLM_ENDPOINT_NAME
evaluation_results['judge_model'] = JUDGE_LLM_ENDPOINT

# Convert to Spark DataFrame
spark_df = spark.createDataFrame(evaluation_results)

# Save to Unity Catalog
table_name = "group4.default.agent_llm_judge_evaluation_results"

print(f"Saving to table: {table_name}")
print(f"Records: {evaluation_results.shape[0]}")
print(f"Columns: {evaluation_results.shape[1]}")

# Write to UC table (append mode to keep history)
spark_df.write.mode("append").saveAsTable(table_name)

print(f"\n✅ Evaluation results saved successfully!")
print(f"\nYou can query the results using:")
print(f"  SELECT * FROM {table_name} WHERE evaluation_timestamp >= current_date()")
print()

# Display summary statistics
print("\n" + "="*80)
print("FINAL SUMMARY STATISTICS")
print("="*80 + "\n")

success_rate = (len(evaluation_results) - evaluation_results['error'].notna().sum()) / len(evaluation_results) * 100
avg_response_time = evaluation_results['response_time'].mean()

print(f"Success Rate: {success_rate:.1f}%")
print(f"Average Response Time: {avg_response_time:.2f}s")

if 'avg_score' in evaluation_results.columns:
    overall_avg = evaluation_results['avg_score'].mean()
    print(f"Overall Average Score: {overall_avg:.2f}/5")
    
    # Score by category
    for metric in ['correctness_score', 'relevance_score', 'groundedness_score']:
        if metric in evaluation_results.columns:
            avg = evaluation_results[metric].mean()
            metric_name = metric.replace('_score', '').title()
            print(f"Average {metric_name}: {avg:.2f}/5")

if 'rejection_score' in evaluation_results.columns:
    rejection_avg = evaluation_results['rejection_score'].mean()
    print(f"\nOut-of-Scope Handling Score: {rejection_avg:.2f}/5")

print(f"\nTotal Traces Generated: {len(evaluation_results)}")
print(f"\nAgent Model: {react_agent.LLM_ENDPOINT_NAME}")
print(f"Judge Model: {JUDGE_LLM_ENDPOINT}")

print("\n" + "="*80)
print("✅ LLM-as-Judge evaluation complete! Results saved and ready for analysis.")
print("="*80)

### Jeffs Code for alignment and judges

In [0]:
# Verify the embeddings table has been populated
emb_df = spark.table("group4.default.ai_reviews_embeddings")
print(f"Embeddings table has {emb_df.count()} rows.")
print(f"Columns: {emb_df.columns}")
display(emb_df.select("App", "Review_Text", "Review_Theme").limit(3))

In [0]:
import mlflow
import sys

# Creating the experiment trace log
EXPERIMENT_NAME = "/Users/" + spark.sql("SELECT current_user()").first()[0] + "/group4_trc_experiment"
mlflow.set_experiment(EXPERIMENT_NAME)

# Workaround for VectorSearchIndex import compatibility issue
# databricks-vectorsearch 0.74 renamed VectorSearchIndex to AISearchIndex
# but databricks_ai_bridge still expects the old name
try:
    from databricks.vector_search.client import AISearchIndex
    sys.modules['databricks.vector_search.client'].VectorSearchIndex = AISearchIndex
except ImportError:
    pass  # If AISearchIndex doesn't exist either, the import will fail with the original error


import agent
AGENT = agent.ToolCallingAgent(llm_endpoint=agent.LLM_ENDPOINT_NAME, tools=agent.TOOL_INFOS)


def _response_to_text(resp):
    """Extract final answer text from ResponsesAgentResponse."""
    text_parts = []
    for item in resp.output:
        c = getattr(item, "content", None)
        if c is None:
            continue
        for part in (c if isinstance(c, list) else [c]):
            if isinstance(part, dict) and "text" in part:
                text_parts.append(part["text"])
            elif hasattr(part, "text"):
                text_parts.append(part.text)
    return "\n".join(text_parts) if text_parts else str(resp.output)[:500]


# Wrapper so evals can use agent_executor.invoke(inputs) -> {"output": "example question"}
class AgentExecutorWrapper:
    def invoke(self, inputs):
        resp = AGENT.predict({
            "input": [{"role": "user", "content": inputs["input"]}],
            "custom_inputs": {"session_id": "final-project-eval"},
        })
        return {"output": _response_to_text(resp)}


agent_executor = AgentExecutorWrapper()

# Quick test
resp = agent_executor.invoke({"input": "How many review theme categories are there?"})
print("Agent test:", resp["output"][:200])

In [0]:
# ---- Manual evaluation questions ----

manual_eval_data = [
    {"inputs": {"question": "What was the most popular review theme?"}},
    {"inputs": {"question": "What was the least popular review theme?"}},
    {"inputs": {"question": "What model had the highest average rating?"}},
    {"inputs": {"question": "What model had the lowest average rating?"}},
    {"inputs": {"question": "Who is the president of the United States?"}},
    {"inputs": {"question": "What is the capital of France?"}},
    {"inputs": {"question": "What was the most common words used in reviews for ChatGPT?"}},
    {"inputs": {"question": "What was the most common feature requested?"}},
    {"inputs": {"question": "Which model has the highest rating?"}},
    {"inputs": {"question": "Highest word count for each model?"}},
    {"inputs": {"question": "How many app versions are there for each model?"}},
    {"inputs": {"question": "Compare the ratings of ChatGPT and Claude."}},
    {"inputs": {"question": "Which model has the best reviews?"}},
    {"inputs": {"question": "Can you create me a chatbot?"}},
    {"inputs": {"question": "What is the average rating for each model?"}}   
]

print(f"Manual eval dataset: {len(manual_eval_data)} questions")

In [0]:
# Combine into the final evaluation dataset
# (If synthetic generation above didn't work, that's OK — the manual set is sufficient)
print(f"Total evaluation dataset: {len(manual_eval_data)} questions")

# Create a UC-backed evaluation dataset and merge records (Databricks pattern:
# https://docs.databricks.com/aws/en/notebooks/source/mlflow3/evaluate-improve-genai-app.html)
uc_schema = "group4.default"
evaluation_dataset_table_name = "group4_eval_dataset"
full_table_name = f"{uc_schema}.{evaluation_dataset_table_name}"
try:
    spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")
except Exception:
    pass
eval_dataset = mlflow.genai.datasets.create_dataset(name=full_table_name)
eval_dataset = eval_dataset.merge_records(manual_eval_data)
# Re-running this cell will append records again; for a fresh dataset, use a new table name or drop the table.
print(f"Evaluation dataset '{full_table_name}' ready ({len(manual_eval_data)} records). Pass eval_dataset to mlflow.genai.evaluate().")

In [0]:
from mlflow.genai.scorers import RelevanceToQuery

# Predict function: one parameter per row so each row = one trace (not one multi-turn conversation).
# Parameter name must match the key in each row's inputs dict (e.g. {"inputs": {"question": "..."}}).
def predict_fn(question: str) -> str:
    """Run the agent on a single question and return the output string."""
    response = agent_executor.invoke({"input": question})
    return response["output"]

# Creating the relevance scorer judge variable
relevance_judge = RelevanceToQuery()

In [0]:
from mlflow.genai.judges import make_judge
from mlflow.genai.scorers import ScorerSamplingConfig

# Custom judge
question_scope_judge = make_judge(
    name="question_scope",  # judge name
    instructions="""You are evaluating the question asked by the user, if the question can not be answered by provided tools then politely reject them as out-of-scope. In-scope questions are those that can be answered by the tools provided. The agent is allowed to use any tool, including the You.com MCP (Databricks). Use {{ inputs }} and {{ outputs }}.



The agent has access to these tools:
- group4__default__all_ai_applications_rating: list all AI applications and their average rating.
- group4__default__fetch_review_theme_category: fetch all review theme categories.
- group4__default__fetch_review_contents: retrieve review content for a specific category, with an optional keyword filter, while limiting the number of records returned per request.
- group4__default__find_most_used_words_by_model: Find the most fequently most used keywords in a prompt grouped by App.

User question: {{inputs}}
Agent response: {{outputs}}

Does the AI agent answer correctly, return True or False.""",
    model="databricks:/databricks-gpt-oss-120b",  # Explicit: judge model gpt-oss-120b
    feedback_value_type=bool,  # Output: bool (True/False)
)

# placeholder if we want to use web search (to be removed)
# , including the You.com MCP (Databricks).
# - You.com MCP (Databricks): live web search


print("This judge is made to evaluate whether the AI agent should answer inscope questions or not.")


# Register the scorer to the current experiment for versioning; Assignment 6 will load it with get_scorer()
# question_scope_judge = question_scope_judge.register()

# ####  Uncomment if re-running this cell  #####
question_scope_judge = question_scope_judge.update(sampling_config=ScorerSamplingConfig(sample_rate=1.0))

In [0]:
# import mlflow
from mlflow.genai import get_scorer

# Load the judge
JUDGE_NAME = "question_scope"  # Judge name
EXPERIMENT_NAME = "/Users/" + spark.sql("SELECT current_user()").first()[0] + "/group4_trc_experiment"

# Get the experiment ID (same experiment as Assignment 5) and load the scorer
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
question_scope_judge = get_scorer(name=JUDGE_NAME, experiment_id=experiment_id)

print(f"Loaded judge '{JUDGE_NAME}' from experiment.")

In [0]:
import mlflow
import sys

# All judges run together.
print("Running evaluation including the all judges...")
all_results = mlflow.genai.evaluate(
    data = eval_dataset,
    predict_fn = predict_fn,
    scorers = [
        relevance_judge,
        question_scope_judge
    ]
)

print("\nEvaluation complete!")

In [0]:
# Agent handling test
resp = agent_executor.invoke({"input": "How hard is python?"})
print("Agent test:", resp["output"][:200])

In [0]:
# Retrieve 15 trace IDs from MLflow; Most recent 15 traces.
# Run again if needing to reset trace IDs.

traces = mlflow.search_traces(
    max_results=15,
    order_by=["timestamp_ms DESC"],
)

print(f"Found {len(traces)} recent traces in experiment.")

In [0]:
# Matching the trace IDs with the answer sheet.

answer_sheet_df = traces.rename(columns={'response': 'output'})  

result = mlflow.genai.evaluate(
    data=answer_sheet_df,
    scorers=[question_scope_judge] 
)

In [0]:
# Tag each trace for alignment for ease of filtering
import mlflow

trace_ids = traces['trace_id'].values

for trace_id in trace_ids:
    mlflow.set_trace_tag(trace_id=trace_id, key="eval", value="align")

print(f"Tagged {len(traces)} traces with eval: align tag.")

In [0]:
# Retrieve the labeled traces that will be used for alignment.
traces_for_alignment = mlflow.search_traces(
    filter_string="tag.eval = 'align'",
    max_results=15,
    return_type="list",
)

# Keep the original trace objects because alignment expects full trace metadata.
print(f"Found {len(traces_for_alignment)} traces with tag eval='align' in experiment.")

In [0]:
# LLM judge assessment and human feedback displayed."Code pulled from AAI 510 HW 6 for displaying results."
def _to_dict_if_possible(obj):
    if isinstance(obj, dict):
        return obj
    for method_name in ("to_dict", "model_dump", "dict"):
        if hasattr(obj, method_name):
            try:
                return getattr(obj, method_name)()
            except Exception:
                pass
    return None


def _get_value(obj, *names):
    if obj is None:
        return None
    if isinstance(obj, dict):
        for name in names:
            if name in obj:
                return obj[name]
        return None
    for name in names:
        if hasattr(obj, name):
            return getattr(obj, name)
    obj_dict = _to_dict_if_possible(obj)
    if isinstance(obj_dict, dict):
        for name in names:
            if name in obj_dict:
                return obj_dict[name]
    return None


def _normalize_text(value):
    if value is None:
        return None
    if isinstance(value, (str, int, float, bool)):
        return str(value)
    value_dict = _to_dict_if_possible(value)
    if isinstance(value_dict, dict):
        for key in ("value", "boolean_value", "string_value", "text"):
            if key in value_dict:
                return str(value_dict[key])
    return str(value)


def _extract_assessments(trace):
    trace_dict = _to_dict_if_possible(trace) or {}
    candidates = [
        _get_value(trace, "assessments"),
        _get_value(_get_value(trace, "info"), "assessments"),
        _get_value(_get_value(trace, "data"), "assessments"),
        trace_dict.get("assessments"),
    ]
    if isinstance(trace_dict.get("info"), dict):
        candidates.append(trace_dict["info"].get("assessments"))
    if isinstance(trace_dict.get("data"), dict):
        candidates.append(trace_dict["data"].get("assessments"))

    for candidate in candidates:
        if candidate:
            return list(candidate)
    return []


def _extract_question(trace):
    trace_dict = _to_dict_if_possible(trace) or {}
    candidates = [
        _get_value(trace, "inputs"),
        _get_value(_get_value(trace, "data"), "inputs"),
        _get_value(_get_value(_get_value(trace, "data"), "request"), "inputs"),
        trace_dict.get("inputs"),
    ]
    if isinstance(trace_dict.get("data"), dict):
        candidates.append(trace_dict["data"].get("inputs"))
        if isinstance(trace_dict["data"].get("request"), dict):
            candidates.append(trace_dict["data"]["request"].get("inputs"))

    for candidate in candidates:
        if isinstance(candidate, dict):
            if "input" in candidate:
                return str(candidate["input"])
            if candidate:
                first_value = next(iter(candidate.values()))
                return str(first_value)
        if isinstance(candidate, str):
            return candidate
    return None


def _extract_trace_id(trace):
    return (
        _get_value(trace, "trace_id")
        or _get_value(_get_value(trace, "info"), "trace_id")
        or (_to_dict_if_possible(trace) or {}).get("trace_id")
        or "unknown-trace"
    )


def _assessment_bucket(assessment):
    source = _get_value(assessment, "source_type", "source", "source_id")
    source_name = _normalize_text(source).lower() if source is not None else ""
    if any(token in source_name for token in ("human", "manual", "feedback", "annotation", "label", "user")):
        return "human"
    return "llm"


def _assessment_summary(assessment):
    value = _normalize_text(_get_value(assessment, "value", "feedback", "result")) or "(missing)"
    rationale = _normalize_text(_get_value(assessment, "rationale", "justification", "explanation")) or "(missing)"
    return value, rationale


print("\n=== Pre-alignment assessment check ===")
llm_present_count = 0
human_present_count = 0
both_present_count = 0
for idx, trace in enumerate(traces_for_alignment, start=1):
    relevant_assessments = [
        assessment
        for assessment in _extract_assessments(trace)
        if (_get_value(assessment, "name", "assessment_name", "key") or JUDGE_NAME) == JUDGE_NAME
    ]

    llm_assessment = next((a for a in relevant_assessments if _assessment_bucket(a) == "llm"), None)
    human_assessment = next((a for a in relevant_assessments if _assessment_bucket(a) == "human"), None)

    question = _extract_question(trace)
    question_preview = (question[:100] + "...") if question and len(question) > 100 else question

    print(f"\nTrace {idx}: {_extract_trace_id(trace)}")
    if question_preview:
        print(f"  Question: {question_preview}")

    llm_value, llm_rationale = _assessment_summary(llm_assessment)
    human_value, human_rationale = _assessment_summary(human_assessment)

    if llm_assessment is not None:
        llm_present_count += 1
    if human_assessment is not None:
        human_present_count += 1
    if llm_assessment is not None and human_assessment is not None:
        both_present_count += 1

    print(f"  LLM assessment: {llm_value}")
    print(f"  LLM rationale: {llm_rationale}")
    print(f"  Human assessment: {human_value}")
    print(f"  Human rationale: {human_rationale}")

print("\n=== Assessment completeness summary ===")
print(f"Traces with LLM assessments: {llm_present_count}/{len(traces_for_alignment)}")
print(f"Traces with human assessments: {human_present_count}/{len(traces_for_alignment)}")
print(f"Traces with both present: {both_present_count}/{len(traces_for_alignment)}")

In [0]:
from mlflow.genai.judges import make_judge
from mlflow.genai.judges.optimizers import SIMBAAlignmentOptimizer

# Aligned the judge with SIMBA.
# Subject Matter Expert feedback using align with SIMBA optimizer.

optimizer = SIMBAAlignmentOptimizer(
    model="databricks:/databricks-gpt-oss-120b",  # Optimizer model used for judge revisions.
    simba_kwargs={
        "max_steps": 2,  # Optimization rounds -> Lower is faster.
        "max_demos": 0,  # Rule-only optimization. Set to greater than 0 letting SIMBA add few-shot demos.
    },
)
aligned_judge = question_scope_judge.align(traces=traces_for_alignment, optimizer=optimizer)
print(f"Alignment complete for judge: {aligned_judge.name}")

In [0]:
print('Question Scope JUDGE:\n')
print(question_scope_judge.instructions)
print('\n')
print("_"*90)
print("\nALIGNED JUDGE\n")
print(aligned_judge.instructions)

In [0]:
print("\nRunning aligned judge for comparison...")
traces_for_eval = mlflow.search_traces(
    filter_string="tag.eval = 'align'",
    max_results=15,
)
df_aligned = traces_for_eval.rename(columns={'response': 'output'}) 

aligned_results = mlflow.genai.evaluate(
    data=df_aligned,
    scorers=[aligned_judge]
)

### End of Human feedback loop code

GEPA Optimizer

## GEPA Optimizer for Question Scope Judge

This section optimizes the `question_scope_judge` instructions using GEPA to improve:
* **Boundary clarity**: Better distinction between in-scope and out-of-scope questions
* **Tool description effectiveness**: Clearer explanation of what each tool can do
* **Edge case handling**: Questions that are partially answerable or ambiguous

In [0]:
%pip install gepa
%pip install litellm
dbutils.library.restartPython()

In [0]:
import gepa

In [0]:
# Create evaluation dataset for the question_scope_judge
# Both in-scope and out-of-scope questions with ground truth labels

judge_eval_dataset = [
    # CLEARLY IN-SCOPE - Questions that tools can answer
    {
        "question": "What is the average rating for all AI applications?",
        "expected_in_scope": True,
        "reason": "all_ai_applications_rating tool provides this"
    },
    {
        "question": "Show me all review theme categories",
        "expected_in_scope": True,
        "reason": "fetch_review_theme_category tool provides this"
    },
    {
        "question": "Find reviews about bugs and crashes",
        "expected_in_scope": True,
        "reason": "fetch_review_contents can filter by theme and keyword"
    },
    {
        "question": "What are the most common words users say about ChatGPT?",
        "expected_in_scope": True,
        "reason": "find_most_used_words_by_model provides keyword frequency"
    },
    
    # CLEARLY OUT-OF-SCOPE - Questions tools cannot answer
    {
        "question": "How hard is Python programming?",
        "expected_in_scope": False,
        "reason": "General knowledge question, no relevant tools"
    },
    {
        "question": "What's the weather like today?",
        "expected_in_scope": False,
        "reason": "Completely unrelated to AI reviews"
    },
    {
        "question": "Tell me a joke about AI",
        "expected_in_scope": False,
        "reason": "Creative request, not data analysis"
    },
    
    # EDGE CASES - Ambiguous questions that need careful judgment
    {
        "question": "Which AI model is best?",
        "expected_in_scope": True,  # Can be answered using ratings
        "reason": "Can infer from average ratings via all_ai_applications_rating"
    },
    {
        "question": "Do users prefer Claude or ChatGPT?",
        "expected_in_scope": True,  # Can compare ratings
        "reason": "Can compare via all_ai_applications_rating"
    },
    {
        "question": "What's the technical architecture of ChatGPT?",
        "expected_in_scope": False,  # Not in review data
        "reason": "Technical details not in user reviews"
    },
    {
        "question": "How do I install ChatGPT?",
        "expected_in_scope": False,  # Not answerable from reviews
        "reason": "Installation instructions not in review data"
    },
]

print(f"Created {len(judge_eval_dataset)} test cases for judge optimization")
print(f"In-scope: {sum(1 for x in judge_eval_dataset if x['expected_in_scope'])}")
print(f"Out-of-scope: {sum(1 for x in judge_eval_dataset if not x['expected_in_scope'])}")

In [0]:
import gepa.optimize_anything as oa
from mlflow.genai.judges import make_judge
from mlflow.genai.scorers import ScorerSamplingConfig


def evaluate_judge_instructions(candidate_instructions: str) -> float:
    """
    Evaluate how well the judge instructions perform on the test dataset.
    
    Args:
        candidate_instructions: The judge instructions to test
    
    Returns:
        float: Score between 0 and 1 (higher is better)
    """
    # Create a judge with the candidate instructions
    test_judge = make_judge(
        name="question_scope_test",
        instructions=candidate_instructions,
        model="databricks:/databricks-gpt-oss-120b",
        feedback_value_type=bool,
    )
    
    # Note: sampling_config not needed for direct evaluate() calls
    
    # Test on all evaluation cases
    correct = 0
    total = len(judge_eval_dataset)
    
    results = []
    
    for test_case in judge_eval_dataset:
        question = test_case["question"]
        expected = test_case["expected_in_scope"]
        
        # Simulate agent response (for judge evaluation)
        mock_output = "I will use the available tools to answer your question."
        
        try:
            # Evaluate with the judge
            result = test_judge.evaluate(
                inputs={"question": question},
                outputs=mock_output
            )
            
            predicted_in_scope = result.feedback_value
            
            if predicted_in_scope == expected:
                correct += 1
                results.append(f"✓ {question[:50]}")
            else:
                results.append(f"✗ {question[:50]} (expected={expected}, got={predicted_in_scope})")
        
        except Exception as e:
            oa.log(f"Error on question: {question[:50]} - {str(e)[:100]}")
            results.append(f"✗ ERROR: {question[:50]}")
    
    accuracy = correct / total
    
    # Log detailed results for GEPA's reflection
    oa.log(f"Accuracy: {accuracy:.2%} ({correct}/{total} correct)")
    oa.log(f"Correctly classified in-scope: {sum(1 for r in results if '✓' in r and 'in-scope' in str(judge_eval_dataset[results.index(r)]['reason']))}")
    oa.log(f"Correctly rejected out-of-scope: {sum(1 for r in results if '✓' in r and 'out-of-scope' in str(judge_eval_dataset[results.index(r)]['expected_in_scope']) and not judge_eval_dataset[results.index(r)]['expected_in_scope'])}")
    
    # Log a few example failures for GEPA's learning
    failures = [r for r in results if '✗' in r]
    if failures:
        oa.log(f"Example failures: {failures[:3]}")
    
    return accuracy

print("✅ GEPA evaluator function defined")

In [0]:
import gepa.optimize_anything as oa
from gepa.optimize_anything import optimize_anything, GEPAConfig, EngineConfig

# Your current judge instructions as the seed
seed_instructions = """You are evaluating the question asked by the user, if the question can not be answered by provided tools then politely reject them as out-of-scope. In-scope questions are those that can be answered by the tools provided. The agent is allowed to use any tool. Use {{ inputs }} and {{ outputs }}.

The agent has access to these tools:
- group4__default__all_ai_applications_rating: list all AI applications and their average rating.
- group4__default__fetch_review_theme_category: fetch all review theme categories.
- group4__default__fetch_review_contents: retrieve review content for a specific category, with an optional keyword filter, while limiting the number of records returned per request.
- group4__default__find_most_used_words_by_model: Finds the 10 most frequently used keywords in the review texts from ai_reviews_embeddings table, grouped by model.

User question: {{inputs}}
Agent response: {{outputs}}

Does the AI agent answer correctly, return True or False."""

print("Starting GEPA optimization for question_scope_judge...")
print(f"Seed instructions accuracy: ", end="")

# Test baseline performance
baseline_score = evaluate_judge_instructions(seed_instructions)
print(f"{baseline_score:.2%}\n")

# Run GEPA optimization
optimization_result = optimize_anything(
    seed_candidate=seed_instructions,
    evaluator=evaluate_judge_instructions,
    objective="Maximize the accuracy of distinguishing in-scope from out-of-scope questions. In-scope questions can be answered using the provided tools (all_ai_applications_rating, fetch_review_theme_category, fetch_review_contents, find_most_used_words_by_model, You.com MCP (Databricks)). Out-of-scope questions are general knowledge, creative requests, or require data not available in the review dataset.",
    config=GEPAConfig(
        engine=EngineConfig(
            max_metric_calls=40,  # Adjust based on time/budget
            display_progress_bar=True
        )
    )
)

print("\n" + "="*80)
print("GEPA OPTIMIZATION COMPLETE")
print("="*80)

# Extract best result from GEPA output
best_idx = optimization_result.val_aggregate_scores.index(max(optimization_result.val_aggregate_scores))
best_score = optimization_result.val_aggregate_scores[best_idx]
best_candidate = optimization_result.candidates[best_idx]

print(f"\nBaseline accuracy: {baseline_score:.2%}")
print(f"Optimized accuracy: {best_score:.2%}")
print(f"Improvement: {(best_score - baseline_score):.2%}")
print(f"\nOptimized instructions:\n{best_candidate}")

In [0]:
# After GEPA optimization, update your question_scope_judge with the best instructions
from mlflow.genai.judges import make_judge

# Replace the seed_instructions with optimization_result.best_candidate
optimized_question_scope_judge = make_judge(
    name="question_scope_optimized",
    instructions=best_candidate['current_candidate'],  # Use GEPA's optimized version
    model="databricks:/databricks-gpt-oss-120b",
    feedback_value_type=bool,
)

print("✅ Updated judge with GEPA-optimized instructions")
print("\nYou can now use 'optimized_question_scope_judge' in your evaluations")

In [0]:
manual_eval_data = [
    {"inputs": {"question": "What was the most popular review theme?"}},
    {"inputs": {"question": "What was the least popular review theme?"}},
    {"inputs": {"question": "What model had the highest average rating?"}},
    {"inputs": {"question": "What model had the lowest average rating?"}},
    {"inputs": {"question": "Who is the president of the United States?"}},
    {"inputs": {"question": "What is the capital of France?"}},
    {"inputs": {"question": "What was the most common words used in reviews for ChatGPT?"}},
    {"inputs": {"question": "What was the most common feature requested?"}},
    {"inputs": {"question": "Which model has the highest rating?"}},
    {"inputs": {"question": "Highest word count for each model?"}},
    {"inputs": {"question": "How many app versions are there for each model?"}},
    {"inputs": {"question": "Compare the ratings of ChatGPT and Claude."}},
    {"inputs": {"question": "Which model has the best reviews?"}},
    {"inputs": {"question": "Can you create me a chatbot?"}},
    {"inputs": {"question": "What is the average rating for each model?"}}   
]

print(f"Manual eval dataset: {len(manual_eval_data)} questions")

In [0]:
# Combine into the final evaluation dataset

print(f"Total evaluation dataset: {len(manual_eval_data)} questions")

# Create a UC-backed evaluation dataset and merge records (Databricks pattern:
# https://docs.databricks.com/aws/en/notebooks/source/mlflow3/evaluate-improve-genai-app.html)
uc_schema = "group4.default"
evaluation_dataset_table_name = "GEPA_group4_eval_dataset"
full_table_name = f"{uc_schema}.{evaluation_dataset_table_name}"
try:
    spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")
except Exception:
    pass
eval_dataset = mlflow.genai.datasets.create_dataset(name=full_table_name)
eval_dataset = eval_dataset.merge_records(manual_eval_data)
# Re-running this cell will append records again; for a fresh dataset, use a new table name or drop the table.
print(f"Evaluation dataset '{full_table_name}' ready ({len(manual_eval_data)} records). Pass eval_dataset to mlflow.genai.evaluate().")

In [0]:
# Register scorer to the current experiment
try:
    optimized_question_scope_judge = optimized_question_scope_judge.register()
except ValueError:
    from mlflow.genai.scorers import ScorerSamplingConfig
    optimized_question_scope_judge = optimized_question_scope_judge.update(sampling_config=ScorerSamplingConfig(sample_rate=1.0))

In [0]:
# Use the optimized judge that was already created and registered in Cell 88 & 92
# No need to load from registry since it's already in memory
question_scope_judge = optimized_question_scope_judge

print(f"Using optimized question_scope_judge: {question_scope_judge.name}")

In [0]:
# import mlflow
from mlflow.genai import get_scorer

# Load the custom judge you created and registered in Assignment 5 (Section 8)
# Use the same JUDGE_NAME you used when you registered the scorer in Assignment 5
JUDGE_NAME = "question_scope"  # Judge name
EXPERIMENT_NAME = "/Users/" + spark.sql("SELECT current_user()").first()[0] + "/GEPA_group4_eval_dataset"

# Get the experiment ID (same experiment as Assignment 5) and load the scorer
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
question_scope_judge = get_scorer(name=JUDGE_NAME, experiment_id=experiment_id)

print(f"Loaded judge '{JUDGE_NAME}' from experiment.")

In [0]:
from mlflow.genai.scorers import RelevanceToQuery

# Predict function: one parameter per row so each row = one trace (not one multi-turn conversation).
# Parameter name must match the key in each row's inputs dict (e.g. {"inputs": {"question": "..."}}).
def predict_fn(question: str) -> str:
    """Run the agent on a single question and return the output string."""
    response = agent_executor.invoke({"input": question})
    return response["output"]

# Recalling the relevance scorer judge variable
relevance_judge = RelevanceToQuery()

In [0]:
# Retrieve 15 trace IDs from MLflow; Most recent 15 traces.

traces = mlflow.search_traces(
    max_results=15,
    order_by=["timestamp_ms DESC"],
)

print(f"Found {len(traces)} recent traces in experiment.")

In [0]:
import mlflow
import sys


# Transform judge_eval_dataset to the format expected by mlflow.genai.evaluate
# MLflow expects data with 'inputs' dict: {"inputs": {"question": "..."}}
eval_data = manual_eval_data

# Single evaluate where all judges run together.
print("Running evaluation including the all judges...")
all_results = mlflow.genai.evaluate(
    data = eval_data,
    predict_fn = predict_fn,
    scorers = [
        relevance_judge,
        question_scope_judge
    ]
)

print("\nEvaluation complete!")

Kept for Human feedback loop code (Should be removed before push)

In [0]:
%pip install -U mlflow
dbutils.library.restartPython()

In [0]:
import mlflow
mlflow.openai.autolog()

In [0]:
# Evaluation Runner
# Executes test queries, collects traces, and evaluates responses

import mlflow
import time
import pandas as pd
from typing import List, Dict, Any

class AgentEvaluator:
    """Comprehensive agent evaluation framework"""
    def __init__(self, agent, judge: LLMJudge):
        self.agent = agent
        self.judge = judge
        self.results = []
    
    def extract_tool_calls(self, response) -> List[str]:
        """Extract tool names from agent response"""
        tool_calls = []
        if hasattr(response, 'output'):
            for item in response.output:
                if hasattr(item, 'type') and item.type == 'function_call':
                    tool_calls.append(item.name)
        return tool_calls
    
    def extract_tool_results(self, response) -> str:
        """Extract tool results from agent response"""
        results = []
        if hasattr(response, 'output'):
            for item in response.output:
                if hasattr(item, 'type') and item.type == 'function_call_output':
                    results.append(f"Tool output: {str(item.output)[:500]}...")  # Truncate long results
        return "\n".join(results) if results else "No tool results"
    
    def evaluate_tool_selection(self, actual_tools: List[str], expected_tools: List[str]) -> Dict[str, Any]:
        """Evaluate if correct tools were selected"""
        actual_set = set(actual_tools)
        expected_set = set(expected_tools)
        
        if len(expected_tools) == 0:  # Out-of-scope query
            return {
                "precision": 1.0 if len(actual_tools) == 0 else 0.0,
                "recall": 1.0,
                "f1": 1.0 if len(actual_tools) == 0 else 0.0,
                "correct": len(actual_tools) == 0
            }
        
        true_positives = len(actual_set & expected_set)
        precision = true_positives / len(actual_set) if len(actual_set) > 0 else 0.0
        recall = true_positives / len(expected_set) if len(expected_set) > 0 else 0.0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        
        return {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "correct": actual_set == expected_set,
            "actual_tools": list(actual_set),
            "expected_tools": list(expected_set),
            "missing_tools": list(expected_set - actual_set),
            "extra_tools": list(actual_set - expected_set)
        }
    
    def run_single_evaluation(self, test_case: Dict[str, Any]) -> Dict[str, Any]:
        """Run evaluation on a single test case"""
        query = test_case["query"]
        query_id = test_case["query_id"]
        query_type = test_case["query_type"]
        
        print(f"\n{'='*60}")
        print(f"Evaluating: [{query_id}] {query}")
        print(f"{'='*60}")
        
        # Execute agent
        start_time = time.time()
        try:
            response = self.agent.predict({
                "input": [{"role": "user", "content": query}],
                "custom_inputs": {"session_id": f"eval-{query_id}"}
            })
            latency_ms = (time.time() - start_time) * 1000
            
            # Extract answer text
            if isinstance(response, dict) and "choices" in response:
                answer = response["choices"][0]["message"]["content"]
            else:
                answer = str(response)
            
            # Extract tool calls and results from response
            actual_tools = self.extract_tool_calls(response)
            tool_results = self.extract_tool_results(response)
            
            # Evaluate tool selection
            tool_eval = self.evaluate_tool_selection(
                actual_tools,
                test_case.get("expected_tools", [])
            )
            
            # LLM judge evaluations
            if query_type == "out_of_scope":
                # For rejection queries
                rejection_eval = self.judge.evaluate_rejection(query, answer)
                correctness_eval = {"score": rejection_eval["score"], "reasoning": "Rejection quality"}
                relevance_eval = {"score": rejection_eval["score"], "reasoning": "N/A for rejection"}
                groundedness_eval = {"score": 5, "reasoning": "N/A for rejection"}
            else:
                # For normal queries
                correctness_eval = self.judge.evaluate_correctness(query, tool_results, answer)
                relevance_eval = self.judge.evaluate_relevance(query, answer)
                groundedness_eval = self.judge.evaluate_groundedness(tool_results, answer)
            
            result = {
                "query_id": query_id,
                "query": query,
                "query_type": query_type,
                "answer": answer,
                "answer_length": len(answer),
                "latency_ms": latency_ms,
                "num_tool_calls": len(actual_tools),
                "actual_tools": actual_tools,
                "expected_tools": test_case.get("expected_tools", []),
                "tool_selection_correct": tool_eval["correct"],
                "tool_selection_f1": tool_eval["f1"],
                "correctness_score": correctness_eval["score"],
                "correctness_reasoning": correctness_eval["reasoning"],
                "relevance_score": relevance_eval["score"],
                "relevance_reasoning": relevance_eval["reasoning"],
                "groundedness_score": groundedness_eval["score"],
                "groundedness_reasoning": groundedness_eval["reasoning"],
                "overall_score": (correctness_eval["score"] + relevance_eval["score"] + groundedness_eval["score"]) / 3,
                "trace_id": f"eval-{query_id}",
                "success": True
            }
            
            print(f"\n✅ Success")
            print(f"   Tools: {actual_tools}")
            print(f"   Correctness: {correctness_eval['score']}/5")
            print(f"   Relevance: {relevance_eval['score']}/5")
            print(f"   Groundedness: {groundedness_eval['score']}/5")
            print(f"   Latency: {latency_ms:.0f}ms")
            
            return result
            
        except Exception as e:
            print(f"\n❌ Error: {str(e)}")
            return {
                "query_id": query_id,
                "query": query,
                "query_type": query_type,
                "error": str(e),
                "success": False,
                "latency_ms": (time.time() - start_time) * 1000
            }
    
    def run_evaluation(self, test_cases: List[Dict[str, Any]]) -> pd.DataFrame:
        """Run evaluation on all test cases"""
        print(f"\n{'#'*60}")
        print(f"# Starting evaluation on {len(test_cases)} test cases")
        print(f"{'#'*60}")
        
        self.results = []
        for test_case in test_cases:
            result = self.run_single_evaluation(test_case)
            self.results.append(result)
            time.sleep(1)  # Brief pause between queries
        
        results_df = pd.DataFrame(self.results)
        return results_df

print("✅ AgentEvaluator class defined")

In [0]:
# Run Evaluation on Test Dataset
# This cell executes the evaluation on all test cases and generates metrics

# Initialize evaluator
evaluator = AgentEvaluator(agent=AGENT, judge=judge)

print("\n" + "="*80)
print("STARTING AGENT EVALUATION")
print("="*80)
print(f"\nAgent: {react_agent.LLM_ENDPOINT_NAME}")
print(f"Judge: {judge.llm_endpoint}")
print(f"Test cases: {len(test_dataset)}")
print(f"\nThis will take approximately {len(test_dataset) * 30} seconds...\n")

# Run evaluation
results_df = evaluator.run_evaluation(test_dataset)

# Display results summary
print("\n" + "="*80)
print("EVALUATION COMPLETE")
print("="*80)

# Show successful evaluations
successful_evals = results_df[results_df['success'] == True]
print(f"\n✅ Successful evaluations: {len(successful_evals)} / {len(results_df)}")

if len(successful_evals) > 0:
    print("\n📊 Summary Statistics:")
    print(f"   Average Correctness Score: {successful_evals['correctness_score'].mean():.2f} / 5")
    print(f"   Average Relevance Score: {successful_evals['relevance_score'].mean():.2f} / 5")
    print(f"   Average Groundedness Score: {successful_evals['groundedness_score'].mean():.2f} / 5")
    print(f"   Average Overall Score: {successful_evals['overall_score'].mean():.2f} / 5")
    print(f"   Average Latency: {successful_evals['latency_ms'].mean():.0f} ms")
    print(f"   Average Tool Calls: {successful_evals['num_tool_calls'].mean():.1f}")
    print(f"   Tool Selection Accuracy: {successful_evals['tool_selection_correct'].mean() * 100:.0f}%")

# Show detailed results
print("\n📋 Detailed Results:")
if len(successful_evals) > 0:
    # Show successful evaluations with full metrics
    display(successful_evals[[
        'query_id', 'query_type', 'num_tool_calls', 'tool_selection_correct',
        'correctness_score', 'relevance_score', 'groundedness_score', 
        'overall_score', 'latency_ms'
    ]])
else:
    # Show all results including errors
    print("⚠️  All evaluations failed. Showing errors:")
    display(results_df[['query_id', 'query_type', 'error', 'latency_ms', 'success']])

# Save results
results_df.to_csv('/Workspace/Users/jssmith@sandiego.edu/Final Project/aai-510-finalproject-group-4/evaluation_results.csv', index=False)
print("\n💾 Results saved to: evaluation_results.csv")

In [0]:
# Detailed Results Analysis
# Deep dive into evaluation results by query type

import matplotlib.pyplot as plt
import numpy as np

print("\n" + "="*80)
print("DETAILED RESULTS ANALYSIS")
print("="*80)

successful_evals = results_df[results_df['success'] == True]

# 1. Performance by Query Type
print("\n1️⃣ Performance by Query Type:")
print("-" * 60)

for query_type in ['factual', 'thematic', 'comparison', 'out_of_scope']:
    type_results = successful_evals[successful_evals['query_type'] == query_type]
    if len(type_results) > 0:
        print(f"\n{query_type.upper()}:")
        print(f"   Count: {len(type_results)}")
        print(f"   Avg Correctness: {type_results['correctness_score'].mean():.2f} / 5")
        print(f"   Avg Relevance: {type_results['relevance_score'].mean():.2f} / 5")
        print(f"   Avg Groundedness: {type_results['groundedness_score'].mean():.2f} / 5")
        print(f"   Tool Selection Accuracy: {type_results['tool_selection_correct'].mean() * 100:.0f}%")
        print(f"   Avg Latency: {type_results['latency_ms'].mean():.0f} ms")

# 2. Tool Selection Analysis
print("\n2️⃣ Tool Selection Analysis:")
print("-" * 60)

correct_tool_calls = successful_evals[successful_evals['tool_selection_correct'] == True]
incorrect_tool_calls = successful_evals[successful_evals['tool_selection_correct'] == False]

print(f"\n✅ Correct Tool Selection: {len(correct_tool_calls)} cases")
if len(correct_tool_calls) > 0:
    print(f"   Avg Score: {correct_tool_calls['overall_score'].mean():.2f} / 5")
    
print(f"\n❌ Incorrect Tool Selection: {len(incorrect_tool_calls)} cases")
if len(incorrect_tool_calls) > 0:
    print(f"   Avg Score: {incorrect_tool_calls['overall_score'].mean():.2f} / 5")
    print("\n   Cases with incorrect tool selection:")
    for _, row in incorrect_tool_calls.iterrows():
        print(f"   - [{row['query_id']}] {row['query'][:50]}...")
        print(f"     Expected: {row['expected_tools']}")
        print(f"     Got: {row['actual_tools']}")

# 3. Top and Bottom Performing Queries
print("\n3️⃣ Top Performing Queries:")
print("-" * 60)
top_queries = successful_evals.nlargest(3, 'overall_score')
for _, row in top_queries.iterrows():
    print(f"\n[{row['query_id']}] Score: {row['overall_score']:.2f}/5")
    print(f"   Query: {row['query']}")
    print(f"   Correctness: {row['correctness_score']}, Relevance: {row['relevance_score']}, Groundedness: {row['groundedness_score']}")

print("\n4️⃣ Lowest Performing Queries:")
print("-" * 60)
bottom_queries = successful_evals.nsmallest(3, 'overall_score')
for _, row in bottom_queries.iterrows():
    print(f"\n[{row['query_id']}] Score: {row['overall_score']:.2f}/5")
    print(f"   Query: {row['query']}")
    print(f"   Correctness: {row['correctness_score']}, Relevance: {row['relevance_score']}, Groundedness: {row['groundedness_score']}")
    print(f"   Issue: {row.get('correctness_reasoning', 'N/A')}")

# 5. Latency Analysis
print("\n5️⃣ Latency Analysis:")
print("-" * 60)
print(f"   Min Latency: {successful_evals['latency_ms'].min():.0f} ms")
print(f"   Max Latency: {successful_evals['latency_ms'].max():.0f} ms")
print(f"   Median Latency: {successful_evals['latency_ms'].median():.0f} ms")
print(f"   95th Percentile: {successful_evals['latency_ms'].quantile(0.95):.0f} ms")

# 6. Rejection Handling Analysis
print("\n6️⃣ Out-of-Scope Query Handling:")
print("-" * 60)
rejection_cases = successful_evals[successful_evals['query_type'] == 'out_of_scope']
if len(rejection_cases) > 0:
    print(f"\n   Total out-of-scope queries: {len(rejection_cases)}")
    print(f"   Avg rejection score: {rejection_cases['correctness_score'].mean():.2f} / 5")
    print(f"   Tool calls made: {rejection_cases['num_tool_calls'].sum()} (should be 0)")
    print("\n   Examples:")
    for _, row in rejection_cases.iterrows():
        print(f"   - [{row['query_id']}] {row['query']}")
        print(f"     Score: {row['correctness_score']}/5, Tools called: {row['num_tool_calls']}")
else:
    print("   No out-of-scope queries in results")

# 7. LLM Comparison + ROI

- Compare 2 LLMs
- Accuracy / quality difference
- Cost difference
- ROI calculation

In [0]:
# 2-LLM Comparison Evaluation
# Compare two different LLM endpoints for the same agent tasks
#
# DEPENDENCIES FROM PREVIOUS CELLS:
# - Cell 60: test_dataset (evaluation test cases)
# - Cell 61: LLMJudge class and judge instance
# - Cell 62: evaluation_results (LLM judge evaluation results)
# - Cell 63: evaluation results summary
# - Cell 64: sample judge reasoning analysis
# - Cell 65: saved evaluation results to Unity Catalog
# - Cell 110: AgentEvaluator class
#
# This setup uses the same test_dataset and evaluation framework
# to compare performance between two different LLM models.

import importlib
import sys

print("\n" + "="*80)
print("2-LLM COMPARISON SETUP")
print("="*80)
print("\nUsing test dataset and evaluation framework from cells 60-65")
print(f"Test cases available: {len(test_dataset) if 'test_dataset' in dir() else 'NOT LOADED'}")
print(f"LLMJudge available: {'YES' if 'LLMJudge' in dir() else 'NO'}")
print(f"AgentEvaluator available: {'YES' if 'AgentEvaluator' in dir() else 'NO'}")

# Define the two LLMs to compare
LLM_A = "databricks-gpt-oss-120b"  # Larger model
LLM_B = "databricks-gpt-oss-20b"   # Smaller model

print(f"\nLLM A (Baseline): {LLM_A}")
print(f"LLM B (Comparison): {LLM_B}")

# Cost assumptions (adjust based on your pricing)
# Cost per 1M tokens (example values - update with actual pricing)
COST_CONFIG = {
    "databricks-gpt-oss-120b": {
        "input_cost_per_1m": 2.143,   # $ per 1M input tokens
        "output_cost_per_1m": 8.571,  # $ per 1M output tokens
    },
    "databricks-gpt-oss-20b": {
        "input_cost_per_1m": 1.00,   # $ per 1M input tokens
        "output_cost_per_1m": 4.286,  # $ per 1M output tokens
    }
}

def estimate_cost(llm_endpoint: str, avg_input_tokens: int, avg_output_tokens: int) -> float:
    """Estimate cost per query in dollars"""
    config = COST_CONFIG.get(llm_endpoint, COST_CONFIG["databricks-gpt-oss-120b"])
    input_cost = (avg_input_tokens / 1_000_000) * config["input_cost_per_1m"]
    output_cost = (avg_output_tokens / 1_000_000) * config["output_cost_per_1m"]
    return input_cost + output_cost

print("\n📊 Cost Configuration (per 1M tokens):")
for llm, config in COST_CONFIG.items():
    print(f"\n{llm}:")
    print(f"   Input: ${config['input_cost_per_1m']:.2f}")
    print(f"   Output: ${config['output_cost_per_1m']:.2f}")

print("\nℹ️  Note: Update COST_CONFIG with actual pricing from this link https://www.databricks.com/product/pricing/foundation-model-serving")

In [0]:
# Run 2-LLM Comparison Evaluation
# Execute the same test cases on both LLMs and compare results

import time

print("\n" + "="*80)
print("RUNNING 2-LLM COMPARISON")
print("="*80)

# Select subset of test cases for comparison (to save time)
comparison_test_cases = [
    test_dataset[0],  # fact_001 - List all AI applications
    test_dataset[1],  # fact_002 - Which app has highest rating
    test_dataset[3],  # theme_001 - What do users complain about bugs
    test_dataset[4],  # theme_002 - Show reviews about crashes
    test_dataset[7],  # reject_001 - Weather question
]

print(f"\nSelected {len(comparison_test_cases)} test cases for comparison")
print("\nTest cases:")
for tc in comparison_test_cases:
    print(f"  - [{tc['query_id']}] {tc['query'][:60]}...")

# Function to run agent with specific LLM
def run_agent_with_llm(llm_endpoint: str, test_cases: list):
    """Create agent with specific LLM and run evaluation"""
    print(f"\n{'='*60}")
    print(f"Testing with: {llm_endpoint}")
    print(f"{'='*60}")
    
    # Create new agent with specified LLM endpoint
    agent_new = react_agent.ToolCallingAgent(
        llm_endpoint=llm_endpoint,
        tools=react_agent.TOOL_INFOS
    )
    
    # Initialize judge with same LLM for consistency
    judge_new = LLMJudge(llm_endpoint)
    
    # Run evaluation
    evaluator_new = AgentEvaluator(agent=agent_new, judge=judge_new)
    results = evaluator_new.run_evaluation(test_cases)
    
    return results

# Run comparison
print("\n🚀 Starting LLM A evaluation...")
results_llm_a = run_agent_with_llm(LLM_A, comparison_test_cases)

print("\n🚀 Starting LLM B evaluation...")
results_llm_b = run_agent_with_llm(LLM_B, comparison_test_cases)

print("\n✅ Both evaluations complete!")

In [0]:
# LLM Comparison Analysis & ROI Calculation
# Compare quality, cost, and ROI between two LLMs

import pandas as pd
import numpy as np

print("\n" + "="*80)
print("LLM COMPARISON ANALYSIS")
print("="*80)

# Filter successful evaluations
llm_a_success = results_llm_a[results_llm_a['success'] == True]
llm_b_success = results_llm_b[results_llm_b['success'] == True]

print(f"\nLLM A ({LLM_A}): {len(llm_a_success)} successful evaluations")
print(f"LLM B ({LLM_B}): {len(llm_b_success)} successful evaluations")

# ===== 1. QUALITY COMPARISON =====
print("\n" + "="*60)
print("1️⃣ QUALITY COMPARISON")
print("="*60)

quality_comparison = pd.DataFrame({
    'Metric': [
        'Correctness Score',
        'Relevance Score',
        'Groundedness Score',
        'Overall Score',
        'Tool Selection Accuracy',
    ],
    f'{LLM_A}': [
        llm_a_success['correctness_score'].mean(),
        llm_a_success['relevance_score'].mean(),
        llm_a_success['groundedness_score'].mean(),
        llm_a_success['overall_score'].mean(),
        llm_a_success['tool_selection_correct'].mean() * 100,
    ],
    f'{LLM_B}': [
        llm_b_success['correctness_score'].mean(),
        llm_b_success['relevance_score'].mean(),
        llm_b_success['groundedness_score'].mean(),
        llm_b_success['overall_score'].mean(),
        llm_b_success['tool_selection_correct'].mean() * 100,
    ]
})

quality_comparison['Difference'] = quality_comparison[f'{LLM_A}'] - quality_comparison[f'{LLM_B}']
quality_comparison['Winner'] = quality_comparison['Difference'].apply(
    lambda x: LLM_A if x > 0 else (LLM_B if x < 0 else 'Tie')
)

print("\n🏆 Quality Metrics (higher is better):")
display(quality_comparison.round(2))

# ===== 2. LATENCY COMPARISON =====
print("\n" + "="*60)
print("2️⃣ LATENCY COMPARISON")
print("="*60)

latency_comparison = pd.DataFrame({
    'Metric': [
        'Average Latency (ms)',
        'Median Latency (ms)',
        'Min Latency (ms)',
        'Max Latency (ms)',
        'P95 Latency (ms)'
    ],
    f'{LLM_A}': [
        llm_a_success['latency_ms'].mean(),
        llm_a_success['latency_ms'].median(),
        llm_a_success['latency_ms'].min(),
        llm_a_success['latency_ms'].max(),
        llm_a_success['latency_ms'].quantile(0.95)
    ],
    f'{LLM_B}': [
        llm_b_success['latency_ms'].mean(),
        llm_b_success['latency_ms'].median(),
        llm_b_success['latency_ms'].min(),
        llm_b_success['latency_ms'].max(),
        llm_b_success['latency_ms'].quantile(0.95)
    ]
})

latency_comparison['Difference (ms)'] = latency_comparison[f'{LLM_A}'] - latency_comparison[f'{LLM_B}']
latency_comparison['Faster'] = latency_comparison['Difference (ms)'].apply(
    lambda x: LLM_B if x > 0 else (LLM_A if x < 0 else 'Tie')
)

print("\n⏱️  Latency Metrics (lower is better):")
display(latency_comparison.round(0))

# ===== 3. COST COMPARISON =====
print("\n" + "="*60)
print("3️⃣ COST COMPARISON")
print("="*60)

# Estimate token usage (approximation: ~3 chars per token for input, ~4 for output)
avg_input_tokens_a = llm_a_success['answer_length'].mean() / 4  # Rough estimate
avg_output_tokens_a = llm_a_success['answer_length'].mean() / 4
avg_input_tokens_b = llm_b_success['answer_length'].mean() / 4
avg_output_tokens_b = llm_b_success['answer_length'].mean() / 4

# For more accurate estimates, use actual context length (query + tool results)
# Assuming average context of ~2000 tokens per query
avg_input_tokens_a = 2000
avg_input_tokens_b = 2000

cost_per_query_a = estimate_cost(LLM_A, avg_input_tokens_a, avg_output_tokens_a)
cost_per_query_b = estimate_cost(LLM_B, avg_input_tokens_b, avg_output_tokens_b)

# Annual cost projection (assuming 1000 queries/day)
queries_per_day = 1000
days_per_year = 365

annual_cost_a = cost_per_query_a * queries_per_day * days_per_year
annual_cost_b = cost_per_query_b * queries_per_day * days_per_year
annual_savings = annual_cost_a - annual_cost_b

cost_comparison_df = pd.DataFrame({
    'Metric': [
        'Cost per Query',
        'Daily Cost (1K queries)',
        'Monthly Cost (30K queries)',
        'Annual Cost (365K queries)'
    ],
    f'{LLM_A}': [
        f"${cost_per_query_a:.6f}",
        f"${cost_per_query_a * queries_per_day:.2f}",
        f"${cost_per_query_a * queries_per_day * 30:.2f}",
        f"${annual_cost_a:.2f}"
    ],
    f'{LLM_B}': [
        f"${cost_per_query_b:.6f}",
        f"${cost_per_query_b * queries_per_day:.2f}",
        f"${cost_per_query_b * queries_per_day * 30:.2f}",
        f"${annual_cost_b:.2f}"
    ],
    'Savings with LLM B': [
        f"${cost_per_query_a - cost_per_query_b:.6f}",
        f"${(cost_per_query_a - cost_per_query_b) * queries_per_day:.2f}",
        f"${(cost_per_query_a - cost_per_query_b) * queries_per_day * 30:.2f}",
        f"${annual_savings:.2f}"
    ]
})

print("\n💰 Cost Analysis:")
display(cost_comparison_df)

# ===== 4. ROI CALCULATION =====
print("\n" + "="*60)
print("4️⃣ ROI ANALYSIS")
print("="*60)

# Quality-adjusted cost (quality score per dollar)
quality_score_a = llm_a_success['overall_score'].mean()
quality_score_b = llm_b_success['overall_score'].mean()

quality_per_dollar_a = quality_score_a / cost_per_query_a if cost_per_query_a > 0 else 0
quality_per_dollar_b = quality_score_b / cost_per_query_b if cost_per_query_b > 0 else 0

roi_a = (quality_score_a / 5) * 100  # Convert to percentage
roi_b = (quality_score_b / 5) * 100

print(f"\n📈 ROI Metrics:")
print(f"\n{LLM_A}:")
print(f"   Quality Score: {quality_score_a:.2f} / 5 ({roi_a:.1f}%)")
print(f"   Cost per Query: ${cost_per_query_a:.6f}")
print(f"   Quality per Dollar: {quality_per_dollar_a:.2f} points/$")
print(f"   Annual Cost (1K queries/day): ${annual_cost_a:.2f}")

print(f"\n{LLM_B}:")
print(f"   Quality Score: {quality_score_b:.2f} / 5 ({roi_b:.1f}%)")
print(f"   Cost per Query: ${cost_per_query_b:.6f}")
print(f"   Quality per Dollar: {quality_per_dollar_b:.2f} points/$")
print(f"   Annual Cost (1K queries/day): ${annual_cost_b:.2f}")

print(f"\n{'='*60}")
print("🎯 RECOMMENDATION:")
print(f"{'='*60}")

quality_diff = quality_score_a - quality_score_b
cost_diff_pct = ((cost_per_query_a - cost_per_query_b) / cost_per_query_a) * 100

if quality_per_dollar_a > quality_per_dollar_b * 1.2:  # 20% better value
    recommendation = f"{LLM_A}"
    reason = f"provides {((quality_per_dollar_a / quality_per_dollar_b - 1) * 100):.0f}% better value (quality per dollar)"
elif quality_per_dollar_b > quality_per_dollar_a * 1.2:
    recommendation = f"{LLM_B}"
    reason = f"provides {((quality_per_dollar_b / quality_per_dollar_a - 1) * 100):.0f}% better value (quality per dollar)"
elif abs(quality_diff) < 0.3:  # Quality is similar
    recommendation = f"{LLM_B}"
    reason = f"provides similar quality ({quality_diff:+.2f} points) at {cost_diff_pct:.0f}% lower cost (${annual_savings:.2f}/year savings)"
else:
    recommendation = f"{LLM_A}"
    reason = f"provides {quality_diff:.2f} points higher quality, worth the additional cost"

print(f"\n⭐ Recommended LLM: {recommendation}")
print(f"   Reason: {reason}")

if annual_savings > 0:
    print(f"\n💵 Potential Annual Savings with {LLM_B}: ${annual_savings:,.2f}")
else:
    print(f"\n💵 Additional Annual Cost with {LLM_A}: ${abs(annual_savings):,.2f}")
    print(f"   Quality improvement: {quality_diff:+.2f} points ({(quality_diff/quality_score_b)*100:+.1f}%)")

In [0]:
# Report Visualizations: Key Metrics Comparison
# Visual summary of LLM comparison results

import matplotlib.pyplot as plt
import numpy as np

fig = plt.figure(figsize=(16, 10))

# Create a 2x3 grid of subplots
gs = fig.add_gridspec(3, 3, hspace=0.4, wspace=0.3)

# Colors
color_120b = '#1f77b4'  # Blue
color_20b = '#ff7f0e'   # Orange

# ===== 1. Quality Metrics Comparison =====
ax1 = fig.add_subplot(gs[0, :])
metrics = ['Correctness', 'Relevance', 'Groundedness', 'Overall\nScore', 'Tool Selection\nAccuracy']

# Use actual data from evaluation results
model_120b_scores = [
    llm_a_success['correctness_score'].mean(),
    llm_a_success['relevance_score'].mean(),
    llm_a_success['groundedness_score'].mean(),
    llm_a_success['overall_score'].mean(),
    llm_a_success['tool_selection_correct'].mean() * 100
]
model_20b_scores = [
    llm_b_success['correctness_score'].mean(),
    llm_b_success['relevance_score'].mean(),
    llm_b_success['groundedness_score'].mean(),
    llm_b_success['overall_score'].mean(),
    llm_b_success['tool_selection_correct'].mean() * 100
]

x = np.arange(len(metrics))
width = 0.35

ax1.bar(x - width/2, model_120b_scores, width, label='120B Model', color=color_120b, alpha=0.8)
ax1.bar(x + width/2, model_20b_scores, width, label='20B Model', color=color_20b, alpha=0.8)
ax1.set_ylabel('Score', fontsize=11)
ax1.set_title('Quality Metrics Comparison (Higher is Better)', fontsize=13, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(metrics, fontsize=10)
ax1.legend(fontsize=10)
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim(0, 100)

# Add value labels on bars
for i, v in enumerate(model_120b_scores):
    ax1.text(i - width/2, v + 2, f'{v:.1f}', ha='center', va='bottom', fontsize=9)
for i, v in enumerate(model_20b_scores):
    ax1.text(i + width/2, v + 2, f'{v:.1f}', ha='center', va='bottom', fontsize=9)

# ===== 2. Latency Comparison =====
ax2 = fig.add_subplot(gs[1, 0])
latency_metrics = ['Avg', 'Median', 'P95']

# Use actual latency data from evaluation results
latency_120b = [
    llm_a_success['latency_ms'].mean(),
    llm_a_success['latency_ms'].median(),
    llm_a_success['latency_ms'].quantile(0.95)
]
latency_20b = [
    llm_b_success['latency_ms'].mean(),
    llm_b_success['latency_ms'].median(),
    llm_b_success['latency_ms'].quantile(0.95)
]

x = np.arange(len(latency_metrics))
ax2.bar(x - width/2, latency_120b, width, label='120B Model', color=color_120b, alpha=0.8)
ax2.bar(x + width/2, latency_20b, width, label='20B Model', color=color_20b, alpha=0.8)
ax2.set_ylabel('Latency (ms)', fontsize=10)
ax2.set_title('Latency Comparison\n(Lower is Better)', fontsize=11, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(latency_metrics, fontsize=9)
ax2.legend(fontsize=9)
ax2.grid(axis='y', alpha=0.3)

# ===== 3. Cost per Query =====
ax3 = fig.add_subplot(gs[1, 1])

# Use actual cost calculations from cell 116
cost_120b = cost_per_query_a
cost_20b = cost_per_query_b

ax3.bar(['120B Model', '20B Model'], [cost_120b, cost_20b], 
        color=[color_120b, color_20b], alpha=0.8, width=0.6)
ax3.set_ylabel('Cost per Query ($)', fontsize=10)
ax3.set_title('Cost per Query\n(Lower is Better)', fontsize=11, fontweight='bold')
ax3.set_ylim(0, 0.045)
ax3.grid(axis='y', alpha=0.3)

# Add value labels
ax3.text(0, cost_120b + 0.001, f'${cost_120b:.4f}', ha='center', va='bottom', fontsize=9)
ax3.text(1, cost_20b + 0.001, f'${cost_20b:.4f}', ha='center', va='bottom', fontsize=9)

# Add savings annotation
savings_pct = ((cost_120b - cost_20b) / cost_120b) * 100
ax3.annotate(f'-{savings_pct:.0f}%\n(${cost_120b - cost_20b:.4f} savings)', 
             xy=(1, cost_20b), xytext=(0.5, cost_120b * 0.9),
             arrowprops=dict(arrowstyle='->', color='green', lw=2),
             fontsize=9, color='green', fontweight='bold',
             ha='center')

# ===== 4. Annual Cost Projection =====
ax4 = fig.add_subplot(gs[1, 2])
timeframes = ['Daily\n(1K)', 'Monthly\n(30K)', 'Annual\n(365K)']

# Use actual cost projections
costs_120b = [
    cost_120b * queries_per_day,
    cost_120b * queries_per_day * 30,
    annual_cost_a
]
costs_20b = [
    cost_20b * queries_per_day,
    cost_20b * queries_per_day * 30,
    annual_cost_b
]

x = np.arange(len(timeframes))
ax4.bar(x - width/2, costs_120b, width, label='120B Model', color=color_120b, alpha=0.8)
ax4.bar(x + width/2, costs_20b, width, label='20B Model', color=color_20b, alpha=0.8)
ax4.set_ylabel('Cost ($)', fontsize=10)
ax4.set_title('Cost Projection by Timeframe', fontsize=11, fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(timeframes, fontsize=9)
ax4.legend(fontsize=9)
ax4.grid(axis='y', alpha=0.3)

# ===== 5. Quality per Dollar (ROI) =====
ax5 = fig.add_subplot(gs[2, 0])

# Use actual quality per dollar calculations from cell 116
quality_per_dollar_120b_val = quality_per_dollar_a
quality_per_dollar_20b_val = quality_per_dollar_b

ax5.bar(['120B Model', '20B Model'], [quality_per_dollar_120b_val, quality_per_dollar_20b_val],
        color=[color_120b, color_20b], alpha=0.8, width=0.6)
ax5.set_ylabel('Quality Points per Dollar', fontsize=10)
ax5.set_title('Quality per Dollar (ROI)\n(Higher is Better)', fontsize=11, fontweight='bold')
max_qpd = max(quality_per_dollar_120b_val, quality_per_dollar_20b_val)
ax5.set_ylim(0, max_qpd * 1.3)
ax5.grid(axis='y', alpha=0.3)

# Add value labels
ax5.text(0, quality_per_dollar_120b_val + max_qpd * 0.02, f'{quality_per_dollar_120b_val:.2f}', 
         ha='center', va='bottom', fontsize=9)
ax5.text(1, quality_per_dollar_20b_val + max_qpd * 0.02, f'{quality_per_dollar_20b_val:.2f}', 
         ha='center', va='bottom', fontsize=9)

# Add improvement annotation
improvement_pct = ((quality_per_dollar_20b_val / quality_per_dollar_120b_val - 1) * 100)
ax5.annotate(f'+{improvement_pct:.0f}%\nbetter value', 
             xy=(1, quality_per_dollar_20b_val), xytext=(0.5, max_qpd * 0.8),
             arrowprops=dict(arrowstyle='->', color='green', lw=2),
             fontsize=9, color='green', fontweight='bold',
             ha='center')

# ===== 6. Annual Savings Breakdown =====
ax6 = fig.add_subplot(gs[2, 1:])

# Stacked bar showing savings (use actual values)
savings_data = {
    '20B Model Cost': annual_cost_b,
    'Annual Savings': annual_savings
}

colors_savings = ['#ff7f0e', '#2ca02c']  # Orange for cost, Green for savings
x_pos = [0]
bottom = 0

for i, (label, value) in enumerate(savings_data.items()):
    ax6.barh(x_pos, value, left=bottom, label=label, 
             color=colors_savings[i], alpha=0.8, height=0.4)
    # Add value label
    ax6.text(bottom + value/2, 0, f'${value:,.0f}', 
             ha='center', va='center', fontsize=10, fontweight='bold', color='white')
    bottom += value

# Add 120B cost reference line
ax6.axvline(annual_cost_a, color=color_120b, linestyle='--', linewidth=2, 
            label='120B Model Cost', alpha=0.7)
ax6.text(annual_cost_a, 0.25, f'120B: ${annual_cost_a:,.0f}', 
         ha='center', fontsize=9, color=color_120b, fontweight='bold')

ax6.set_xlabel('Annual Cost ($)', fontsize=10)
ax6.set_title('Annual Cost Comparison @ 1K queries/day\n(Savings = 120B Cost - 20B Cost)', 
              fontsize=11, fontweight='bold')
ax6.set_yticks([])
ax6.legend(loc='upper right', fontsize=9)
ax6.grid(axis='x', alpha=0.3)
ax6.set_xlim(0, annual_cost_a * 1.1)

# Add title for entire figure
fig.suptitle('LLM Model Comparison: 120B vs 20B\nQuality, Performance, Cost & ROI Analysis', 
             fontsize=15, fontweight='bold', y=0.995)

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("📊 KEY TAKEAWAYS FROM VISUALIZATIONS")
print("="*80)
# Print actual comparison summary
quality_diff_pct = ((quality_score_a - quality_score_b) / quality_score_b) * 100
tool_diff_pct = ((llm_a_success['tool_selection_correct'].mean() - llm_b_success['tool_selection_correct'].mean()) / llm_b_success['tool_selection_correct'].mean()) * 100
latency_diff_pct = ((llm_a_success['latency_ms'].mean() - llm_b_success['latency_ms'].mean()) / llm_a_success['latency_ms'].mean()) * 100
cost_diff_pct = ((cost_120b - cost_20b) / cost_120b) * 100
roi_diff_pct = ((quality_per_dollar_20b_val / quality_per_dollar_120b_val - 1) * 100)

print(f"\n✅ QUALITY: 120B model scores {quality_diff_pct:+.1f}% higher overall ({quality_score_a:.2f} vs {quality_score_b:.2f})")
print(f"   ➤ 120B has {tool_diff_pct:+.0f}% better tool selection accuracy ({llm_a_success['tool_selection_correct'].mean()*100:.0f}% vs {llm_b_success['tool_selection_correct'].mean()*100:.0f}%)")
print(f"\n⚡ PERFORMANCE: 120B model is {abs(latency_diff_pct):.0f}% {'faster' if latency_diff_pct < 0 else 'slower'} on average")
print(f"   ➤ Avg latency: {llm_a_success['latency_ms'].mean():.0f}ms vs {llm_b_success['latency_ms'].mean():.0f}ms")
print(f"\n💰 COST: 20B model costs {cost_diff_pct:.0f}% less (${cost_20b:.4f} vs ${cost_120b:.4f} per query)")
print(f"   ➤ Annual savings: ${annual_savings:,.0f} at 1K queries/day")
print(f"\n🎯 ROI: 20B model delivers {roi_diff_pct:.0f}% better value (quality per dollar)")
print(f"   ➤ {quality_per_dollar_20b_val:.2f} vs {quality_per_dollar_120b_val:.2f} quality points per dollar")
print(f"\n🏆 RECOMMENDATION: {recommendation}")
print(f"   Reason: {reason}")
print("="*80)

# 8. Limitations, Deployment, and Reflection

- What worked well
- What was weak
- Human evaluation role
- Deployment recommendation
- AI tool disclosure